# Genex Interview + Activity Brain - v13

This notebook is the cleaned working notebook for the Genex developmental interview + home-support planning prototype for children ages 0-5.

Version 11 keeps the prior architecture — including **subdomain-aware concern routing**, **split internal language scoring**, broad **risk-based safety constraints**, and **three-stage developmental band scoring** — and adds two product-facing planning improvements:

1. **Family guidance floor when everything is technically no special support**
   - The core support tiers are still:
     - **no_special_support**
     - **monitor_and_enrich**
     - **needs_special_support**
   - But if every category lands at **no_special_support**, the system now still creates a light **enrich_and_observe** plan in the strongest concern-relevant category.
   - This keeps the product parent-friendly: families still leave with short, age-appropriate enrichment ideas and an explicit monitor-and-observe message, rather than leaving with no activities at all.

This is intended to be a **soft floor for family guidance**, not a hidden override of milestone evidence:
- milestone evidence still drives the support tier
- the enrich-and-observe layer only activates when all categories are otherwise no special support
- the selected guidance category is based on the strongest concern signal, with age-appropriate low-intensity activities

2. **Bonus-only rule for playdate / park / group activities**
   - Core weekly schedule now stays focused on **short, repeatable, at-home activities**.
   - Context-dependent activities such as **playdates, park meetups, playground peer activities, or group participation** are treated as **optional weekly bonus activities only**.
   - These bonus activities are automatically moved out of the main day-by-day schedule and normalized to **30-45 minutes**.

The notebook still supports both:
- **live parent mode**
- **Gemini synthetic-parent mode**

and still exports advisor review packets, rating sheets, and JSON payloads.


## Recommended run order

1. Install cell  
2. Imports + environment setup  
3. Config  
4. CDC loading + normalization  
5. Core helpers + concern router + safety profile  
6. Delay estimation  
7. Milestone engine  
8. Q&A mode (live or Gemini)  
9. Support tier + activity generation + weekly scheduling  
10. Main runners  
11. Sanity check  
12. One test case  
13. Batch export if needed  

### Notes for v13
- The notebook still uses the CDC milestone table as the structured backbone.
- If the CDC workbook contains an **`advisor_supplemental_review`** sheet with rows marked for inclusion, those approved supplemental items can still be appended automatically.
- v10 keeps the v9 scoring changes:
  - safer **within-domain language scoring**
  - **confirmed / emerging / not_demonstrated** month-band scoring
  - stronger **risk-based activity safety constraints**
  - motor-aware emerging scoring
- The new v13 changes include a **family guidance floor**:
  - if all categories are `no_special_support`, the notebook still builds a short **enrich_and_observe** plan
  - the plan is assigned to the strongest concern-relevant category
  - the activities are age-appropriate, low-intensity, and framed as enrichment + monitoring rather than special support
- Synthetic-parent mode still requires a working Gemini API key and quota/billing.

- v13 also enforces a **bonus-only rule** for context-dependent activities:
  - playdates, park meetups, playground peer activities, and group participation are not placed in the main daily schedule
  - these are allowed only in **Optional Weekly Bonus Activity**
  - they are normalized to **30-45 minutes** rather than appearing as unrealistic 5-minute tasks


In [ ]:
# Install once per notebook kernel if needed
%pip install -U pandas openpyxl python-dotenv openai google-genai reportlab


In [ ]:
%pip install -U google-genai

In [ ]:
import os
from openai import OpenAI
from google import genai

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY is missing.")
if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY is missing.")

openai_client = OpenAI(api_key=OPENAI_API_KEY)
gemini_client = genai.Client(api_key=GEMINI_API_KEY)

print("OpenAI visible:", bool(OPENAI_API_KEY))
print("Gemini visible:", bool(GEMINI_API_KEY))

In [ ]:
# ------------------------------------------------------------
# Imports + environment setup
# ------------------------------------------------------------
import os
import re
import json
import time
from pathlib import Path
from datetime import datetime
from typing import Any, Dict, List, Optional

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

try:
    from openai import OpenAI
except ImportError:
    OpenAI = None

try:
    from google import genai
except ImportError:
    genai = None

from reportlab.lib.pagesizes import landscape, LETTER
from reportlab.lib.units import inch
from reportlab.pdfgen import canvas
from reportlab.pdfbase.pdfmetrics import stringWidth

load_dotenv()

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "genex_interview_activity_v9"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

In [ ]:
# ------------------------------------------------------------
# Config
# ------------------------------------------------------------
DOMAIN_CONFIG = {
    "movement_and_physical": {
        "display": "Movement / Physical",
        "short": "motor",
    },
    "social_and_emotional": {
        "display": "Social / Emotional",
        "short": "social_emotional",
    },
    "language_and_communication": {
        "display": "Language / Communication",
        "short": "language_communication",
    },
    "cognitive": {
        "display": "Cognitive / Adaptive",
        "short": "cognitive",
    },
}

ALIAS_TO_CATEGORY = {
    "movement and physical": "movement_and_physical",
    "physical": "movement_and_physical",
    "motor": "movement_and_physical",
    "gross motor": "movement_and_physical",

    "social and emotional": "social_and_emotional",
    "social and emotial": "social_and_emotional",
    "social_emotional": "social_and_emotional",
    "social": "social_and_emotional",

    "language and communication": "language_and_communication",
    "language": "language_and_communication",
    "speech": "language_and_communication",
    "speech and language": "language_and_communication",

    "cognitive": "cognitive",
    "adaptive": "cognitive",
}

ANSWER_SCORES = {
    "yes": 1.0,
    "sometimes": 0.4,
    "with_help": 0.2,
    "no": 0.0,
    "not_sure": 0.1,
}

VALID_ANSWERS = set(ANSWER_SCORES.keys())
VALID_PARENT_ANSWERS = {"yes", "sometimes", "with_help", "no", "not_sure"}


FOLLOWUP_SCHEMAS = {
    "sometimes": {
        "prompt": "Why only sometimes? Choose the best fit:",
        "choices": [
            ("not_consistent_yet", "can do it, but not consistently yet"),
            ("distracted", "can do it, but gets distracted"),
            ("upset_or_refuses", "can do it, but gets upset or refuses"),
            ("only_some_situations", "can do it, but only in some situations"),
            ("not_sure", "not sure"),
        ],
    },
    "with_help": {
        "prompt": "What kind of help is usually needed?",
        "choices": [
            ("physical_help", "physical help / hands-on support"),
            ("reminders_prompting", "reminders or step-by-step prompting"),
            ("emotional_support", "encouragement or emotional support"),
            ("showing_first", "showing first / demonstrating"),
            ("not_sure", "not sure"),
        ],
    },
    "no": {
        "prompt": "Can you tell us a little more about the 'no'? Choose the best fit:",
        "choices": [
            ("not_able_yet", "not able yet"),
            ("does_not_do_even_when_we_try", "does not do it even when we try"),
            ("upset_or_refuses", "gets upset or refuses"),
            ("distracted_before_doing", "gets distracted before doing it"),
            ("not_sure_why", "not sure why"),
        ],
    },
}

PERFORMANCE_BARRIER_MAP = {
    ("yes", ""): "none",
    ("not_sure", ""): "unclear",

    ("sometimes", "not_consistent_yet"): "skill_emerging",
    ("sometimes", "distracted"): "distractibility",
    ("sometimes", "upset_or_refuses"): "emotional_dysregulation_refusal",
    ("sometimes", "only_some_situations"): "situational_inconsistency",
    ("sometimes", "not_sure"): "unclear",

    ("with_help", "physical_help"): "physical_help",
    ("with_help", "reminders_prompting"): "needs_prompting",
    ("with_help", "emotional_support"): "emotional_support",
    ("with_help", "showing_first"): "needs_demonstration",
    ("with_help", "not_sure"): "unclear",

    ("no", "not_able_yet"): "not_able_yet",
    ("no", "does_not_do_even_when_we_try"): "does_not_do_even_when_we_try",
    ("no", "upset_or_refuses"): "emotional_dysregulation_refusal",
    ("no", "distracted_before_doing"): "distractibility",
    ("no", "not_sure_why"): "unclear",
}


# In v9, all domains use a 3-stage band idea:
# confirmed / emerging / not_demonstrated.
#
# Emerging is allowed everywhere, but it is weighted more strongly in motor/postural
# subdomains because motor skills are often acquired gradually and inconsistently before
# stable mastery.
MOTOR_EMERGING_SUBDOMAINS = {
    "postural_control_and_transitions",
    "gross_motor_mobility_and_coordination",
    "fine_motor_hand_use",
}

GENERAL_EMERGING_PARTIAL_WEIGHT = 0.45
GENERAL_EMERGING_NO_PENALTY = 0.40

MOTOR_EMERGING_PARTIAL_WEIGHT = 0.70
MOTOR_EMERGING_NO_PENALTY = 0.25


In [ ]:

# ------------------------------------------------------------
# CDC loading + normalization + optional advisor supplemental sheet
# ------------------------------------------------------------
PREFERRED_CDC_FILENAMES = [
    "milestone-cdc-table-improved-subdomains-advisor.xlsx",
    "milestone-cdc-table-improved-subdomains.xlsx",
    "milestone-cdc-table.xlsx",
    "milestone-cdc-table(2).xlsx",
]
ADVISOR_SUPPLEMENTAL_SHEET_NAME = "advisor_supplemental_review"

def find_cdc_file(path: Optional[str] = None) -> Path:
    """Find the CDC milestone table, preferring the improved subdomain-tagged version."""
    if path:
        path = Path(path)
        if not path.exists():
            raise FileNotFoundError(f"Provided CDC file path does not exist: {path}")
        return path.resolve()

    search_roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent, Path("/mnt/data")]

    for name in PREFERRED_CDC_FILENAMES:
        for root in search_roots:
            candidate = root / name
            if candidate.exists():
                return candidate.resolve()

    candidate_paths = []
    for root in search_roots:
        if root.exists():
            candidate_paths.extend(root.rglob("*milestone*cdc*table*.xlsx"))
            candidate_paths.extend(root.rglob("*milestone*subdomain*.xlsx"))

    candidate_paths = list({p.resolve() for p in candidate_paths if p.exists()})
    if candidate_paths:
        def rank_path(p: Path):
            name = p.name.lower()
            if "subdomain" in name or "improved" in name:
                return (0, len(name))
            if "(2)" in name:
                return (2, len(name))
            return (1, len(name))
        candidate_paths = sorted(candidate_paths, key=rank_path)
        return candidate_paths[0]

    raise FileNotFoundError(
        "Could not find a CDC milestone spreadsheet. Put the improved subdomain-tagged file in the repo root, "
        "notebooks parent folder, or /mnt/data."
    )

def _normalize_cdc_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).strip().lower() for c in df.columns]
    df = df.rename(columns={"category ": "category", "milestone ": "milestone"})

    if "subdomain" not in df.columns:
        df["subdomain"] = "unspecified"

    df["category"] = df["category"].astype(str).str.strip().str.lower()
    df["milestone"] = df["milestone"].astype(str).str.strip()
    df["subdomain"] = df["subdomain"].fillna("unspecified").astype(str).str.strip().str.lower()
    df["months"] = pd.to_numeric(df["months"], errors="coerce")

    df = df.dropna(subset=["months", "category", "milestone"]).copy()
    df["months"] = df["months"].astype(int)
    df["category_key"] = df["category"].map(lambda x: ALIAS_TO_CATEGORY.get(x, x.replace(" ", "_")))
    df["question_id"] = [
        f"{row.category_key}_{row.months}_{i}"
        for i, row in enumerate(df.itertuples(), start=1)
    ]
    return df

def _normalize_supplemental_sheet(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).strip().lower() for c in df.columns]

    rename_map = {
        "main_category": "category",
        "target_subdomain": "subdomain",
        "proposed_supplemental_milestone_or_followup": "milestone",
        "include_in_model_after_review": "include_in_model",
    }
    df = df.rename(columns=rename_map)

    for col in ["category", "subdomain", "milestone", "include_in_model"]:
        if col not in df.columns:
            df[col] = None
    if "months" not in df.columns:
        df["months"] = None

    df["category"] = df["category"].astype(str).str.strip().str.lower()
    df["subdomain"] = df["subdomain"].astype(str).str.strip().str.lower()
    df["milestone"] = df["milestone"].astype(str).str.strip()
    df["include_in_model"] = df["include_in_model"].astype(str).str.strip().str.lower()
    df["months"] = pd.to_numeric(df["months"], errors="coerce")

    include_values = {"yes", "approved", "include", "1", "true", "y"}
    df = df[
        df["include_in_model"].isin(include_values)
        & df["months"].notna()
        & df["category"].astype(bool)
        & df["milestone"].astype(bool)
        & df["subdomain"].astype(bool)
    ].copy()

    if df.empty:
        return df

    df["months"] = df["months"].astype(int)
    df["category_key"] = df["category"].map(lambda x: ALIAS_TO_CATEGORY.get(x, x.replace(" ", "_")))
    df["question_id"] = [
        f"{row.category_key}_{row.months}_supp_{i}"
        for i, row in enumerate(df.itertuples(), start=1)
    ]
    df["source"] = "advisor_supplemental"
    return df[["months", "category", "milestone", "subdomain", "category_key", "question_id", "source"]]

def load_cdc_table(path: Optional[str] = None):
    """Load the CDC backbone and, if present, append any advisor-approved supplemental items.

    The main interview still uses the CDC table as the backbone.
    v7 simply provides a hook so a future reviewed supplemental sheet can be included
    without replacing the CDC source.
    """
    path = find_cdc_file(path)
    xls = pd.ExcelFile(path)

    main_sheet = "all" if "all" in xls.sheet_names else xls.sheet_names[0]
    base_df = pd.read_excel(xls, sheet_name=main_sheet)
    base_df = _normalize_cdc_dataframe(base_df)
    base_df["source"] = "cdc_backbone"

    approved_supplemental_df = pd.DataFrame(columns=base_df.columns)

    if ADVISOR_SUPPLEMENTAL_SHEET_NAME in xls.sheet_names:
        supplemental_raw = pd.read_excel(xls, sheet_name=ADVISOR_SUPPLEMENTAL_SHEET_NAME)
        approved_supplemental_df = _normalize_supplemental_sheet(supplemental_raw)

        if not approved_supplemental_df.empty:
            missing_cols = [c for c in base_df.columns if c not in approved_supplemental_df.columns]
            for c in missing_cols:
                approved_supplemental_df[c] = None
            approved_supplemental_df = approved_supplemental_df[base_df.columns]

    combined_df = pd.concat([base_df, approved_supplemental_df], ignore_index=True, sort=False)
    combined_df = combined_df.sort_values(["months", "category", "milestone"]).reset_index(drop=True)

    return combined_df, path, approved_supplemental_df

cdc_df, cdc_path, approved_supplemental_df = load_cdc_table()
CDC_AGES = sorted(cdc_df["months"].dropna().unique().tolist())

print("Loaded CDC file:", cdc_path)
print("CDC ages:", CDC_AGES)
print("Rows in combined milestone bank:", len(cdc_df))
print("Rows from CDC backbone:", int((cdc_df.get("source", "") == "cdc_backbone").sum()))
print("Approved supplemental rows loaded:", len(approved_supplemental_df))
print("Has subdomain column:", "subdomain" in cdc_df.columns)
print("Unique subdomains:", cdc_df["subdomain"].nunique())

def get_category_questions(category_key: str, min_months: int, max_months: int) -> pd.DataFrame:
    subset = cdc_df[
        (cdc_df["category_key"] == category_key)
        & (cdc_df["months"] >= min_months)
        & (cdc_df["months"] <= max_months)
    ].sort_values(["months", "milestone"])
    return subset.copy()

display(cdc_df.head())


In [ ]:

# ------------------------------------------------------------
# Core helpers + state + concern router
# ------------------------------------------------------------
def normalize_answer(answer_text: str) -> str:
    """Normalize a raw parent answer into the allowed answer set."""
    t = str(answer_text).strip().lower().replace(" ", "_")
    if t in VALID_ANSWERS:
        return t
    if t in {"notsure", "unsure", "maybe"}:
        return "not_sure"
    return "not_sure"


def get_followup_schema(answer_norm: str):
    return FOLLOWUP_SCHEMAS.get(str(answer_norm).strip().lower())

def followup_label_from_key(answer_norm: str, key: str) -> str:
    schema = get_followup_schema(answer_norm)
    if not schema:
        return ""
    for choice_key, label in schema["choices"]:
        if choice_key == key:
            return label
    return ""

def normalize_followup_answer(answer_norm: str, raw_text: str) -> str:
    schema = get_followup_schema(answer_norm)
    if not schema:
        return ""

    raw = str(raw_text).strip().lower()
    if not raw:
        return ""

    if raw.isdigit():
        idx = int(raw)
        if 1 <= idx <= len(schema["choices"]):
            return schema["choices"][idx - 1][0]

    compact = raw.replace(" ", "_").replace("/", "_").replace("-", "_")
    compact = re.sub(r"_+", "_", compact).strip("_")

    valid_keys = {k for k, _ in schema["choices"]}
    if compact in valid_keys:
        return compact

    for key, label in schema["choices"]:
        if raw == label.lower():
            return key
        if compact == re.sub(r"_+", "_", label.lower().replace(" ", "_").replace("/", "_").replace("-", "_")).strip("_"):
            return key
        if raw in label.lower():
            return key

    return "not_sure" if "not_sure" in valid_keys else schema["choices"][-1][0]

def infer_performance_barrier(answer_norm: str, followup_key: str = "") -> str:
    answer_norm = str(answer_norm).strip().lower()
    followup_key = str(followup_key).strip().lower()
    if (answer_norm, followup_key) in PERFORMANCE_BARRIER_MAP:
        return PERFORMANCE_BARRIER_MAP[(answer_norm, followup_key)]
    if (answer_norm, "") in PERFORMANCE_BARRIER_MAP:
        return PERFORMANCE_BARRIER_MAP[(answer_norm, "")]
    return "unclear"

def ask_live_followup(answer_norm: str):
    schema = get_followup_schema(answer_norm)
    if not schema:
        return "", ""
    print(schema["prompt"])
    for idx, (_, label) in enumerate(schema["choices"], start=1):
        print(f"  {idx}. {label}")
    raw = input("Type the number or phrase that fits best: ").strip()
    followup_key = normalize_followup_answer(answer_norm, raw)
    return followup_key, followup_label_from_key(answer_norm, followup_key)


def print_banner(title: str):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

def extract_json_block(text: str) -> dict:
    """Parse strict JSON or recover a simple answer from loosely formatted model text."""
    text = str(text).strip()

    try:
        return json.loads(text)
    except Exception:
        pass

    m = re.search(r"```json\s*(\{.*?\})\s*```", text, flags=re.DOTALL)
    if m:
        try:
            return json.loads(m.group(1))
        except Exception:
            pass

    m = re.search(r"(\{.*\})", text, flags=re.DOTALL)
    if m:
        try:
            return json.loads(m.group(1))
        except Exception:
            pass

    lowered = text.lower()
    for label in ["with_help", "not_sure", "sometimes", "yes", "no"]:
        if label in lowered:
            return {"answer": label, "reason": text[:200]}

    if "with help" in lowered:
        return {"answer": "with_help", "reason": text[:200]}
    if "not sure" in lowered or "unsure" in lowered or "maybe" in lowered:
        return {"answer": "not_sure", "reason": text[:200]}

    raise ValueError(f"Could not parse JSON from model output: {text[:300]}")

def new_state() -> Dict[str, Any]:
    """Initialize the working state dictionary for one case."""
    return {
        "child": {},
        "concern_profile": {},
        "delay_estimates": {},
        "qna": {},
        "dev_age": {},
        "activity_banks": {},
        "weekly_slot_allocation": {},
        "weekly_schedule": {},
        "safety_profile": {},
    }

SUBDOMAIN_TO_CATEGORY = {}
for subdomain, group in cdc_df.groupby("subdomain"):
    cats = [c for c in group["category_key"].dropna().astype(str).unique().tolist() if c]
    if cats:
        SUBDOMAIN_TO_CATEGORY[subdomain] = cats[0]

CATEGORY_TO_SUBDOMAINS = {
    category_key: sorted(
        cdc_df.loc[cdc_df["category_key"] == category_key, "subdomain"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )
    for category_key in DOMAIN_CONFIG
}

SUBDOMAIN_KEYWORD_MAP = {
    "speech_intelligibility": [
        r"\bapraxia\b",
        r"\bchildhood apraxia of speech\b",
        r"\bcas\b",
        r"hard to understand",
        r"unclear speech",
        r"inconsistent sounds",
        r"speech output",
        r"speech hard to understand",
        r"articulation",
        r"motor speech",
        r"intelligib",
    ],
    "expressive_language": [
        r"no words",
        r"few words",
        r"limited speech",
        r"limited words",
        r"only\s*~?\d+\s*words",
        r"very limited speech",
        r"speech delay",
        r"late talk",
        r"two[- ]word",
        r"phrases",
        r"naming",
    ],
    "receptive_language": [
        r"understands well",
        r"good comprehension",
        r"comprehension",
        r"doesn't understand",
        r"does not understand",
        r"follows? simple directions",
        r"follows? directions",
        r"commands",
        r"receptive",
    ],
    "gestural_communication": [
        r"uses gestures",
        r"gesture",
        r"pointing",
        r"\bpoints?\b",
        r"signs?",
        r"communicat",
    ],
    "early_vocalization_and_babbling": [
        r"limited babbling",
        r"\bbabbl",
        r"\bcoo",
        r"vocal",
        r"raspberr",
        r"squeal",
        r"makes sounds",
    ],
    "conversation_narrative": [
        r"conversation",
        r"answers questions",
        r"tell(s)? (a )?story",
        r"narrative",
    ],
    "social_engagement_and_joint_attention": [
        r"eye contact",
        r"limited eye contact",
        r"joint attention",
        r"shared attention",
        r"responds? to name",
        r"responds? to her name",
        r"responds? to his name",
        r"socially engaged",
        r"good eye contact",
    ],
    "peer_interaction_and_social_rules": [
        r"\bpeer",
        r"friends?",
        r"play with children",
        r"plays with other children",
        r"takes? turns?",
        r"interrupts",
        r"social but rigid",
        r"social rules",
    ],
    "play_and_symbolic_social_play": [
        r"pretend play",
        r"limited pretend play",
        r"symbolic play",
        r"repetitive play",
        r"lines up toys",
    ],
    "emotional_regulation": [
        r"tantrum",
        r"meltdown",
        r"big emotional reactions",
        r"gets frustrated",
        r"frustrat",
        r"emotional regulation",
        r"self-regulation",
        r"impulsive",
        r"very active",
        r"hyperactive",
        r"adhd",
    ],
    "attachment_and_separation": [
        r"clingy",
        r"separation",
        r"drop off",
        r"leave(s|) the room",
        r"when you leave",
    ],
    "empathy_and_prosocial_behavior": [
        r"empathy",
        r"notices when others are hurt",
        r"kind to others",
        r"hurt or upset",
    ],
    "gross_motor_mobility_and_coordination": [
        r"not walking",
        r"walking",
        r"frequent falls",
        r"stairs",
        r"balance",
        r"clumsy gait",
        r"\bgait\b",
        r"run",
        r"jump",
        r"walker",
        r"mobility",
    ],
    "postural_control_and_transitions": [
        r"not sitting",
        r"sitting independently",
        r"sit independently",
        r"head control",
        r"hypotonia",
        r"posture",
        r"rolling",
        r"get to sitting",
        r"pushes up",
    ],
    "fine_motor_hand_use": [
        r"fine motor",
        r"grasp",
        r"beads?",
        r"string",
        r"\bfork\b",
        r"crayon",
        r"pencil",
        r"hand use",
    ],
    "self_help_motor_skills": [
        r"self-care",
        r"dress",
        r"clothes",
        r"buttons?",
        r"zippers?",
        r"utensil",
        r"\bspoon\b",
    ],
    "adaptive_feeding_cues": [
        r"slow feeding",
        r"feeding",
        r"picky eating",
        r"oral motor",
        r"open mouth",
        r"close lips",
    ],
    "attention_and_processing": [
        r"attention",
        r"short attention span",
        r"\bfocus\b",
        r"processing",
    ],
    "concepts_and_following_directions": [
        r"follow(s|) directions",
        r"one-step",
        r"two-step",
        r"concepts?",
        r"colors?",
        r"letters?",
        r"numbers?",
    ],
    "exploration_and_object_use": [
        r"explore",
        r"object use",
        r"toy use",
        r"puts things in (his|her|their) mouth",
    ],
    "imitation_and_play_skills": [
        r"imitat",
        r"\bcopy",
        r"copies",
        r"play skills",
    ],
    "object_permanence_and_problem_solving": [
        r"problem solving",
        r"finds?",
        r"hides?",
        r"object permanence",
    ],
    "pre_academic_skills": [
        r"letters?",
        r"count(ing)?",
        r"colors?",
        r"pre-academic",
    ],
    "safety_awareness": [
        r"safety",
        r"danger",
        r"unsafe",
    ],
}

def _apply_concern_propagation(subdomain_weights: Dict[str, float]) -> Dict[str, float]:
    """Lightly propagate strong signals into related subdomains.

    This keeps the concern router systematic while avoiding brittle one-off hacks.
    """
    weights = dict(subdomain_weights)

    if weights.get("speech_intelligibility", 0) >= 0.60:
        weights["expressive_language"] = max(weights.get("expressive_language", 0), 0.70)

    if weights.get("expressive_language", 0) >= 0.60 and weights.get("gestural_communication", 0) >= 0.40:
        weights["early_vocalization_and_babbling"] = max(weights.get("early_vocalization_and_babbling", 0), 0.40)

    if weights.get("emotional_regulation", 0) >= 0.60:
        weights["concepts_and_following_directions"] = max(weights.get("concepts_and_following_directions", 0), 0.40)

    if weights.get("gross_motor_mobility_and_coordination", 0) >= 0.60:
        weights["postural_control_and_transitions"] = max(weights.get("postural_control_and_transitions", 0), 0.40)

    if weights.get("adaptive_feeding_cues", 0) >= 0.60:
        weights["self_help_motor_skills"] = max(weights.get("self_help_motor_skills", 0), 0.30)

    return weights


POSITIVE_ROUTING_HINTS = [
    "good eye contact",
    "good comprehension",
    "understands well",
    "good cognition",
    "socially engaged",
    "strong language skills",
    "strong language",
    "very verbal",
]

def _pattern_match_weight(pattern_text: str) -> float:
    pattern_text = str(pattern_text).lower()
    if any(hint in pattern_text for hint in POSITIVE_ROUTING_HINTS):
        return 0.18
    return 0.35

def concern_router(child: Dict[str, Any]) -> Dict[str, Any]:
    """Convert diagnosis + concern text into structured subdomain and domain weights.

    v6 keeps this deterministic and inspectable. The goal is not to diagnose from the concern,
    but to create a weak, transparent routing signal that can help question selection and
    borderline support decisions.
    """
    diagnosis = str(child.get("diagnosis", "") or "")
    concern = str(child.get("concern", "") or "")
    combined_text = f"{diagnosis} | {concern}".lower()

    subdomain_weights = {s: 0.0 for s in SUBDOMAIN_TO_CATEGORY.keys()}
    matched_patterns = {s: [] for s in SUBDOMAIN_TO_CATEGORY.keys()}

    for subdomain, patterns in SUBDOMAIN_KEYWORD_MAP.items():
        matches = []
        for pat in patterns:
            if re.search(pat, combined_text):
                matches.append(pat)

        if matches:
            weight = min(1.0, sum(_pattern_match_weight(pat) for pat in matches))
            subdomain_weights[subdomain] = max(subdomain_weights.get(subdomain, 0.0), weight)
            matched_patterns[subdomain] = matches

    subdomain_weights = _apply_concern_propagation(subdomain_weights)

    domain_weights = {k: 0.0 for k in DOMAIN_CONFIG.keys()}
    for subdomain, weight in subdomain_weights.items():
        category_key = SUBDOMAIN_TO_CATEGORY.get(subdomain)
        if category_key in domain_weights:
            domain_weights[category_key] = max(domain_weights[category_key], float(weight))

    top_subdomains = [
        {"subdomain": s, "weight": round(w, 2)}
        for s, w in sorted(subdomain_weights.items(), key=lambda kv: kv[1], reverse=True)
        if w > 0
    ][:8]

    return {
        "combined_text": combined_text,
        "subdomain_weights": subdomain_weights,
        "domain_weights": domain_weights,
        "matched_patterns": matched_patterns,
        "top_subdomains": top_subdomains,
    }

def ensure_concern_profile(state: Dict[str, Any]) -> Dict[str, Any]:
    """Compute and cache the concern profile if needed."""
    if not state.get("concern_profile"):
        state["concern_profile"] = concern_router(state["child"])
    state["safety_profile"] = build_safety_profile(state["child"])
    return state["concern_profile"]

def print_concern_profile(state: Dict[str, Any]) -> None:
    """Print a compact, human-readable concern-routing summary."""
    profile = ensure_concern_profile(state)

    print("\n" + "=" * 100)
    print("CONCERN ROUTER PROFILE")
    print("=" * 100)

    domain_weights = profile.get("domain_weights", {})
    top_domains = [
        (DOMAIN_CONFIG[k]["display"], round(v, 2))
        for k, v in sorted(domain_weights.items(), key=lambda kv: kv[1], reverse=True)
        if v > 0
    ]
    if top_domains:
        print("Domain weights:")
        for name, weight in top_domains:
            print(f"- {name}: {weight}")
    else:
        print("Domain weights: no strong routed concern signals detected.")

    top_subdomains = profile.get("top_subdomains", [])
    if top_subdomains:
        print("Top subdomains:")
        for item in top_subdomains[:6]:
            print(f"- {item['subdomain']}: {item['weight']}")

def print_child_reference(state: Dict[str, Any]) -> None:
    child = state["child"]
    print("\n" + "=" * 100)
    print("CHILD REFERENCE CHECK")
    print("=" * 100)
    print(f"Name: {child.get('name', '')}")
    print(f"Chronological age: {child.get('chronological_months', '')} months")
    print(f"Diagnosis / condition: {child.get('diagnosis', '')}")
    print(f"Parent concern: {child.get('concern', '')}")
    print(f"Daily time available: {child.get('daily_time_min', '')} minutes")

def intake_agent_live(state: Dict[str, Any]) -> Dict[str, Any]:
    """Collect a live child profile from user input."""
    print_banner("INTAKE AGENT")

    name = input("What is your child's name? ").strip()
    chronological_months = int(
        input("How old is your child in months? (for example: 18, 24, 36, 48, 60) ").strip()
    )
    diagnosis = input("Any diagnosis / condition? (type 'none' if none) ").strip()
    concern = input("What are your main concerns right now? ").strip()
    daily_time_min = int(
        input("About how many minutes per day can you usually spend helping or playing with your child? ").strip()
    )

    state["child"] = {
        "name": name,
        "chronological_months": chronological_months,
        "diagnosis": diagnosis,
        "concern": concern,
        "daily_time_min": daily_time_min,
    }
    state["concern_profile"] = concern_router(state["child"])
    state["safety_profile"] = build_safety_profile(state["child"])
    return state["child"]

def init_state_from_case(case: Dict[str, Any], daily_time_min: int = 10) -> Dict[str, Any]:
    """Initialize a state object from a prefilled synthetic case."""
    state = new_state()
    state["child"] = {
        "name": case["child_name"],
        "chronological_months": int(case["age_months"]),
        "diagnosis": case["diagnosis"],
        "concern": case["concerns"],
        "daily_time_min": int(daily_time_min),
    }
    state["concern_profile"] = concern_router(state["child"])
    state["safety_profile"] = build_safety_profile(state["child"])
    return state


SAFETY_KEYWORD_MAP = {
    "falls_balance_gait": [
        r"frequent falls",
        r"fall",
        r"difficulty with stairs",
        r"stairs",
        r"clumsy gait",
        r"balance",
        r"walker",
        r"unsteady",
        r"cerebral palsy",
        r"ataxia",
    ],
    "postural_low_tone_fatigue": [
        r"hypotonia",
        r"low muscle tone",
        r"low tone",
        r"tires quickly",
        r"fatigue",
        r"weak",
        r"not sitting independently",
        r"not walking",
    ],
    "fine_motor_or_coordination": [
        r"fine motor",
        r"grip",
        r"coordination",
        r"clumsy hand",
        r"uses a fork",
        r"beads",
        r"crayon",
        r"pencil",
    ],
    "feeding_or_oral_motor": [
        r"feeding",
        r"slow feeding",
        r"chew",
        r"swallow",
        r"chok",
        r"drool",
        r"oral motor",
        r"oral-motor",
    ],
    "regulation_frustration": [
        r"tantrum",
        r"frustrat",
        r"difficulty with transitions",
        r"transitions",
        r"rigid",
        r"big emotional reactions",
        r"impuls",
        r"short attention",
        r"attention",
        r"adhd",
    ],
    "sensory_sensitivity": [
        r"sensory",
        r"sensitive",
        r"picky eating",
        r"texture",
        r"noise",
        r"overwhelm",
    ],
    "mobility_equipment_support": [
        r"walker",
        r"wheelchair",
        r"braces",
        r"orthotic",
        r"uses support",
    ],
    "seizure_or_medical_monitoring": [
        r"seizure",
        r"epilep",
        r"drop attack",
        r"medical frag",
    ],
    "high_activity_open_space_risk": [
        r"very active",
        r"bolts",
        r"runs off",
        r"elopes",
        r"unsafe climbing",
    ],
}

SAFETY_CONSTRAINT_TEMPLATES = {
    "falls_balance_gait": "Keep activities ground-level and closely supervised. Avoid high surfaces, jumping from heights, or tasks that assume stable balance without support.",
    "postural_low_tone_fatigue": "Prefer short bouts with rest breaks, supported positioning, and lower-endurance tasks. Avoid long sustained postures beyond the child's tolerance.",
    "fine_motor_or_coordination": "Adapt hand demands with larger tools/items, stabilizing support, and slower pacing rather than precision-heavy expectations.",
    "feeding_or_oral_motor": "Keep feeding or oral-motor tasks upright and closely supervised. Avoid choking-risk foods or unsafe oral-motor suggestions.",
    "regulation_frustration": "Keep tasks short, predictable, and easy to start. Build in transitions, choices, and stop before escalation or overload.",
    "sensory_sensitivity": "Use low-clutter, lower-noise materials when possible and introduce textures/sounds gradually.",
    "mobility_equipment_support": "Adapt tasks for walker/braces/other supports and do not assume unsupported stairs, standing, or walking.",
    "seizure_or_medical_monitoring": "Avoid activities that would be unsafe if sudden loss of awareness or control occurred. Keep supervision explicit and setups low risk.",
    "high_activity_open_space_risk": "Use contained spaces, clear physical boundaries, and active supervision. Avoid tasks that depend on open unsafe spaces or long waiting.",
}


def build_safety_profile(child: Dict[str, Any]) -> Dict[str, Any]:
    """Build a broad safety/practical constraint profile from diagnosis + concern text.

    v9 keeps the broad, reusable risk profile from v7 but now adds:
    - explicit hard-avoid lists for clearly risky activity types
    - preferred adaptation notes that can be passed directly into prompts
    - a deterministic post-generation safety pass downstream
    """
    diagnosis = str(child.get("diagnosis", "") or "")
    concern = str(child.get("concern", "") or "")
    combined_text = f"{diagnosis} | {concern}".lower()

    risk_scores = {k: 0.0 for k in SAFETY_KEYWORD_MAP.keys()}
    matched_patterns = {k: [] for k in SAFETY_KEYWORD_MAP.keys()}

    for risk_name, patterns in SAFETY_KEYWORD_MAP.items():
        matches = []
        for pat in patterns:
            if re.search(pat, combined_text):
                matches.append(pat)
        if matches:
            risk_scores[risk_name] = min(1.0, 0.35 * len(matches))
            matched_patterns[risk_name] = matches

    top_risks = [
        {"risk": k, "weight": round(v, 2)}
        for k, v in sorted(risk_scores.items(), key=lambda kv: kv[1], reverse=True)
        if v > 0
    ][:6]

    constraints = [
        SAFETY_CONSTRAINT_TEMPLATES[risk_name]
        for risk_name, score in sorted(risk_scores.items(), key=lambda kv: kv[1], reverse=True)
        if score >= 0.35 and risk_name in SAFETY_CONSTRAINT_TEMPLATES
    ]

    hard_avoid = []
    preferred_adaptations = []

    high_fall_or_mobility = (
        risk_scores.get("falls_balance_gait", 0.0) >= 0.35
        or risk_scores.get("mobility_equipment_support", 0.0) >= 0.35
    )
    if high_fall_or_mobility:
        hard_avoid.extend([
            "jumping from heights or unsupported jumping drills",
            "trampoline-style activities",
            "playground climbing or high unstable surfaces",
            "unsupported balance challenges on unstable surfaces",
        ])
        preferred_adaptations.extend([
            "prefer ground-level or seated versions when possible",
            "use hand support, wall support, or equipment support as needed",
            "favor supported reaching, weight shifting, step-up, and transfer practice over high-risk movement tasks",
        ])

    if risk_scores.get("postural_low_tone_fatigue", 0.0) >= 0.35:
        hard_avoid.extend([
            "long sustained postures without rest",
            "high-endurance tasks that assume good postural endurance",
        ])
        preferred_adaptations.extend([
            "short bouts with rest breaks",
            "supported positioning and lower-endurance versions",
        ])

    if risk_scores.get("feeding_or_oral_motor", 0.0) >= 0.35:
        hard_avoid.extend([
            "unsafe oral-motor suggestions",
            "choking-risk foods or unsupervised feeding tasks",
        ])
        preferred_adaptations.extend([
            "keep feeding upright and closely supervised",
            "use safer textures and stop if coughing, gagging, or distress occurs",
        ])

    if risk_scores.get("regulation_frustration", 0.0) >= 0.35:
        hard_avoid.extend([
            "long multi-step activities without breaks",
            "activities that depend on long waiting or repeated failure without support",
        ])
        preferred_adaptations.extend([
            "keep tasks short, predictable, and easy to start",
            "offer choices and stop before escalation",
        ])

    if risk_scores.get("sensory_sensitivity", 0.0) >= 0.35:
        preferred_adaptations.extend([
            "use lower-noise and lower-clutter materials when possible",
            "introduce textures and sounds gradually",
        ])

    # de-duplicate while preserving order
    def _dedupe(items):
        seen = set()
        out = []
        for x in items:
            if x not in seen:
                seen.add(x)
                out.append(x)
        return out

    constraints = _dedupe(constraints)
    hard_avoid = _dedupe(hard_avoid)
    preferred_adaptations = _dedupe(preferred_adaptations)

    if not constraints:
        constraints = ["No major safety constraints were inferred from the diagnosis/concern text. Keep activities parent-supervised and home-safe."]

    return {
        "combined_text": combined_text,
        "risk_scores": risk_scores,
        "matched_patterns": matched_patterns,
        "top_risks": top_risks,
        "constraints": constraints,
        "hard_avoid": hard_avoid,
        "preferred_adaptations": preferred_adaptations,
    }

def ensure_safety_profile(state: Dict[str, Any]) -> Dict[str, Any]:
    if not state.get("safety_profile"):
        state["safety_profile"] = build_safety_profile(state["child"])
    return state["safety_profile"]

def format_safety_constraints_for_prompt(profile: Dict[str, Any]) -> str:
    lines = []
    constraints = profile.get("constraints", [])
    hard_avoid = profile.get("hard_avoid", [])
    adaptations = profile.get("preferred_adaptations", [])

    if constraints:
        lines.append("Planning constraints:")
        lines.extend([f"- {c}" for c in constraints])

    if hard_avoid:
        lines.append("Hard avoidances:")
        lines.extend([f"- {c}" for c in hard_avoid])

    if adaptations:
        lines.append("Preferred adaptations:")
        lines.extend([f"- {c}" for c in adaptations])

    if not lines:
        lines = ["- No special safety constraints flagged. Keep activities parent-supervised and home-safe."]

    return "\n".join(lines)

def print_safety_profile(state: Dict[str, Any]) -> None:
    profile = ensure_safety_profile(state)
    print("\n" + "=" * 100)
    print("SAFETY / PRACTICAL CONSTRAINT PROFILE")
    print("=" * 100)
    top_risks = profile.get("top_risks", [])
    if not top_risks:
        print("No elevated safety/practical flags detected from diagnosis/concern text.")
        return
    print("Top inferred risks:")
    for item in top_risks:
        print(f"- {item['risk']}: {item['weight']}")
    print("Planning constraints:")
    for c in profile.get("constraints", []):
        print(f"- {c}")
    if profile.get("hard_avoid"):
        print("Hard avoidances:")
        for c in profile.get("hard_avoid", []):
            print(f"- {c}")
    if profile.get("preferred_adaptations"):
        print("Preferred adaptations:")
        for c in profile.get("preferred_adaptations", []):
            print(f"- {c}")

def print_safety_profile(state: Dict[str, Any]) -> None:
    profile = ensure_safety_profile(state)
    print("\n" + "=" * 100)
    print("SAFETY / PRACTICAL CONSTRAINT PROFILE")
    print("=" * 100)
    top_risks = profile.get("top_risks", [])
    if not top_risks:
        print("No elevated safety/practical flags detected from diagnosis/concern text.")
        return
    print("Top inferred risks:")
    for item in top_risks:
        print(f"- {item['risk']}: {item['weight']}")
    print("Planning constraints:")
    for c in profile.get("constraints", []):
        print(f"- {c}")


In [ ]:
# ------------------------------------------------------------
# Delay estimation
# ------------------------------------------------------------
def estimate_delay_gpt(
    diagnosis: str,
    concern: str,
    chronological_months: int,
    category_key: str,
    model: str = "gpt-4o-mini",
) -> Dict[str, Any]:
    """Estimate a rough starting delay in months for one domain.

    This is only a starting anchor for question selection, not the final developmental age.
    """
    category_display = DOMAIN_CONFIG[category_key]["display"]
    concern_l = (concern or "").lower()

    fallback_by_category = {
        "movement_and_physical": 3,
        "language_and_communication": 3,
        "social_and_emotional": 6,
        "cognitive": 6,
    }

    domain_keywords = {
        "movement_and_physical": [
            "motor", "movement", "walk", "run", "jump", "balance", "coordination",
            "fine motor", "gross motor", "grasp", "hand", "writing", "stairs", "falls",
            "hypotonia", "sitting", "rolling", "crawling"
        ],
        "language_and_communication": [
            "speech", "language", "talk", "communication", "words", "sentence",
            "understand", "expressive", "receptive", "verbal", "babbling", "no words"
        ],
        "social_and_emotional": [
            "social", "peer", "friend", "play", "emotion", "emotional",
            "behavior", "anger", "meltdown", "interaction", "turn taking",
            "regulation", "eye contact", "transitions", "bullied", "bullying"
        ],
        "cognitive": [
            "attention", "focus", "concentration", "school", "learning", "routine",
            "executive", "task", "independent", "adaptive", "toilet", "dressing",
            "self-care", "directions", "paying attention"
        ],
    }

    has_domain_signal = any(kw in concern_l for kw in domain_keywords.get(category_key, []))

    if openai_client is None:
        delay_months = fallback_by_category.get(category_key, 6)
        if not has_domain_signal:
            delay_months = min(delay_months, 6)
        return {
            "delay_months": delay_months,
            "reason": f"Fallback delay estimate used for {category_display} because the OpenAI client is not available.",
            "source": "fallback",
        }

    prompt = f"""
You are a pediatric developmental delay estimator agent for children ages 0 to 5 years.

Your job is to estimate a SINGLE STARTING DELAY in months for one developmental domain only.

This is NOT a diagnosis.
This is NOT a final developmental age.
This is ONLY a rough starting anchor for question selection.

Definition:
delay_months = chronological age in months - estimated functional developmental age in this specific domain

Child information:
- Chronological age in months: {chronological_months}
- Diagnosis / condition: {diagnosis}
- Parent concern: {concern}
- Domain to estimate: {category_display}

Instructions:
1. Think only about THIS domain, not overall development.
2. If the diagnosis or concern does NOT meaningfully affect this domain, return 0 to 6 months.
3. If this domain IS affected, estimate the child's functional developmental level in this domain, then convert it into delay_months.
4. Be conservative but realistic.
5. Never exceed the child's chronological age.
6. Return only one integer number of months.
7. Return strict JSON only.

Required JSON:
{{
  "delay_months": <integer>,
  "reason": "<one short sentence>"
}}
""".strip()

    try:
        resp = openai_client.chat.completions.create(
            model=model,
            temperature=0.2,
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": "You return strict JSON only."},
                {"role": "user", "content": prompt},
            ],
        )

        parsed = json.loads(resp.choices[0].message.content)
        delay_months = int(parsed.get("delay_months", fallback_by_category.get(category_key, 6)))
        delay_months = max(0, min(delay_months, chronological_months))

        if not has_domain_signal and delay_months > 6:
            delay_months = 6

        return {
            "delay_months": delay_months,
            "reason": parsed.get("reason", ""),
            "source": "openai",
        }

    except Exception as e:
        return {
            "delay_months": fallback_by_category.get(category_key, 6),
            "reason": f"Fallback delay estimate used because the OpenAI call failed: {e}",
            "source": "fallback",
        }

def delay_agent_all_categories(state: Dict[str, Any], categories: Optional[List[str]] = None) -> Dict[str, Any]:
    """Estimate a starting delay for each category."""
    print_banner("DELAY ESTIMATOR AGENT")
    if not state.get("child"):
        raise ValueError("Child profile missing. Run intake first.")

    categories = categories or list(DOMAIN_CONFIG.keys())
    child = state["child"]

    for category_key in categories:
        est = estimate_delay_gpt(
            diagnosis=child["diagnosis"],
            concern=child["concern"],
            chronological_months=child["chronological_months"],
            category_key=category_key,
        )
        state["delay_estimates"][category_key] = est
        print(f"{DOMAIN_CONFIG[category_key]['display']}: {est['delay_months']} month starting delay | {est['reason']}")

    return state["delay_estimates"]


In [ ]:

# ------------------------------------------------------------
# Core milestone engine
# ------------------------------------------------------------
def _age_proximity_score(month: int, approx_dev_months: int, window_months: int) -> float:
    """Score how close a month band is to the current estimated developmental age."""
    denom = max(window_months / 2, 1)
    score = 1.0 - abs(month - approx_dev_months) / denom
    return float(max(0.0, min(1.0, score)))

def build_milestone_questions(
    state: Dict[str, Any],
    category_key: str,
    window_months: int = 24,
    target_questions_per_band: int = 3,
    max_bands: int = 3,
    max_questions_total: int = 9,
) -> List[Dict[str, Any]]:
    """Build a focused set of milestone questions.

    v6 still centers the interview around the estimated developmental range, but it now uses
    the concern router as a *bias* within that range rather than asking the same generic
    questions for every case in the same broad category.
    """
    child = state["child"]
    concern_profile = ensure_concern_profile(state)

    chrono_months = min(child["chronological_months"], 60)
    delay_months = state["delay_estimates"].get(category_key, {}).get("delay_months", 12)

    approx_dev_months = max(2, chrono_months - delay_months)
    min_months = max(2, approx_dev_months - window_months // 2)
    max_months = min(60, approx_dev_months + window_months // 2)

    subset = get_category_questions(category_key, min_months=min_months, max_months=max_months)
    if subset.empty:
        subset = get_category_questions(category_key, min_months=min(CDC_AGES), max_months=max(CDC_AGES))

    if subset.empty:
        return []

    subset = subset.copy()
    if "subdomain" not in subset.columns:
        subset["subdomain"] = "unspecified"

    category_concern_weight = concern_profile["domain_weights"].get(category_key, 0.0)

    subset["age_score"] = subset["months"].map(lambda m: _age_proximity_score(int(m), approx_dev_months, window_months))
    subset["subdomain_weight"] = subset["subdomain"].map(
        lambda s: concern_profile["subdomain_weights"].get(str(s), 0.0)
    )
    subset["row_score"] = (
        0.60 * subset["age_score"] +
        0.30 * subset["subdomain_weight"] +
        0.10 * float(category_concern_weight)
    )

    available_months = sorted(subset["months"].unique().tolist())
    base_months = sorted(
        available_months,
        key=lambda m: (abs(m - approx_dev_months), m)
    )[:max_bands]

    selected_months = list(base_months)

    # Concern-aware month injection:
    # if the concern strongly points to a subdomain that lives in a slightly different month band,
    # we allow one targeted band to replace the weakest/farthest band.
    strong_rows = subset[subset["subdomain_weight"] >= 0.55].sort_values(
        ["subdomain_weight", "row_score", "months"], ascending=[False, False, True]
    )
    if category_concern_weight >= 0.50 and not strong_rows.empty:
        concern_month = int(strong_rows.iloc[0]["months"])
        if concern_month not in selected_months and selected_months:
            farthest_base = max(selected_months, key=lambda m: (abs(m - approx_dev_months), m))
            selected_months.remove(farthest_base)
            selected_months.append(concern_month)

    selected_months = sorted(set(selected_months))

    questions = []
    for month in selected_months:
        month_rows = subset[subset["months"] == month].sort_values(
            ["row_score", "milestone"], ascending=[False, True]
        )
        month_rows = month_rows.head(target_questions_per_band)

        for _, row in month_rows.iterrows():
            questions.append({
                "question_id": row["question_id"],
                "months": int(row["months"]),
                "milestone": row["milestone"],
                "subdomain": str(row.get("subdomain", "unspecified")),
                "selection_score": round(float(row.get("row_score", 0.0)), 3),
                "question_text": f"Can {child['name']} {row['milestone']} right now? (yes / sometimes / with_help / no / not_sure)",
            })

    questions = sorted(questions, key=lambda q: (q["months"], q["selection_score"] * -1))
    return questions[:max_questions_total]


def _band_has_motor_emphasis(items: List[Dict[str, Any]]) -> bool:
    if not items:
        return False
    subdomains = [str(x.get("subdomain", "unspecified")).strip().lower() for x in items]
    motor_count = sum(1 for s in subdomains if s in MOTOR_EMERGING_SUBDOMAINS)
    return motor_count / max(len(subdomains), 1) >= 0.50


def classify_band_stage(
    *,
    total: int,
    yes_count: int,
    partial_count: int,
    no_count: int,
    motor_emphasis: bool,
    min_yes_confirm: int = 2,
    yes_ratio_confirm: float = 0.60,
) -> Dict[str, Any]:
    """Classify a month band into confirmed / emerging / not_demonstrated.

    Version 9 keeps one shared 3-stage framework across all domains, but allows
    emerging performance to count more strongly in motor/postural subdomains because
    those skills often appear gradually and inconsistently before stable mastery.
    """
    yes_ratio = yes_count / total if total else 0.0
    positive_ratio = (yes_count + partial_count) / total if total else 0.0

    if yes_count >= min_yes_confirm and yes_ratio >= yes_ratio_confirm:
        return {
            "stage": "confirmed",
            "motor_emphasis": motor_emphasis,
            "evidence_score": round(2.0 + yes_ratio, 3),
            "yes_ratio": round(yes_ratio, 2),
            "positive_ratio": round(positive_ratio, 2),
        }

    partial_weight = MOTOR_EMERGING_PARTIAL_WEIGHT if motor_emphasis else GENERAL_EMERGING_PARTIAL_WEIGHT
    no_penalty = MOTOR_EMERGING_NO_PENALTY if motor_emphasis else GENERAL_EMERGING_NO_PENALTY

    evidence_score = (
        float(yes_count) * 1.0 +
        float(partial_count) * partial_weight -
        float(no_count) * no_penalty
    )

    emerging = False

    # Shared emerging logic: visible partial or mixed evidence can indicate developmental level.
    if positive_ratio >= 0.67 and (yes_count > 0 or partial_count >= 2):
        emerging = True

    # Stronger emerging interpretation for motor/postural bands.
    if motor_emphasis:
        if partial_count >= 2 and no_count <= 1:
            emerging = True
        if partial_count == total and total >= 2:
            emerging = True

    if emerging:
        return {
            "stage": "emerging",
            "motor_emphasis": motor_emphasis,
            "evidence_score": round(float(evidence_score), 3),
            "yes_ratio": round(yes_ratio, 2),
            "positive_ratio": round(positive_ratio, 2),
        }

    return {
        "stage": "not_demonstrated",
        "motor_emphasis": motor_emphasis,
        "evidence_score": round(float(evidence_score), 3),
        "yes_ratio": round(yes_ratio, 2),
        "positive_ratio": round(positive_ratio, 2),
    }


def summarize_answers_by_band(
    answers: List[Dict[str, Any]],
    min_yes_confirm: int = 2,
    yes_ratio_confirm: float = 0.60,
) -> Dict[int, Dict[str, Any]]:
    """Summarize answers by month band for debugging and advisor review.

    Version 9 adds a 3-stage band label:
    - confirmed
    - emerging
    - not_demonstrated

    Emerging is available to all domains, but it is weighted more strongly for
    motor/postural subdomains.
    """
    band_summary = {}

    for a in answers:
        month = int(a["months"])
        norm = a["norm_answer"]

        if month not in band_summary:
            band_summary[month] = {
                "total": 0,
                "yes": 0,
                "partial": 0,
                "no": 0,
                "items": [],
            }

        band_summary[month]["total"] += 1
        band_summary[month]["items"].append({
            "milestone": a["milestone"],
            "subdomain": a.get("subdomain", "unspecified"),
            "answer": norm,
        })

        if norm == "yes":
            band_summary[month]["yes"] += 1
        elif norm in {"sometimes", "with_help", "not_sure"}:
            band_summary[month]["partial"] += 1
        else:
            band_summary[month]["no"] += 1

    for month in band_summary:
        total = band_summary[month]["total"]
        yes_count = band_summary[month]["yes"]
        partial_count = band_summary[month]["partial"]
        no_count = band_summary[month]["no"]

        motor_emphasis = _band_has_motor_emphasis(band_summary[month]["items"])
        stage_info = classify_band_stage(
            total=total,
            yes_count=yes_count,
            partial_count=partial_count,
            no_count=no_count,
            motor_emphasis=motor_emphasis,
            min_yes_confirm=min_yes_confirm,
            yes_ratio_confirm=yes_ratio_confirm,
        )

        band_summary[month]["yes_ratio"] = round(yes_count / total, 2) if total else 0.0
        band_summary[month]["stage"] = stage_info["stage"]
        band_summary[month]["motor_emphasis"] = motor_emphasis
        band_summary[month]["evidence_score"] = stage_info["evidence_score"]
        band_summary[month]["positive_ratio"] = stage_info["positive_ratio"]

    return dict(sorted(band_summary.items()))


def compute_dev_age_from_answers(
    answers: List[Dict[str, Any]],
    min_yes_confirm: int = 2,
    yes_ratio_confirm: float = 0.60,
) -> int:
    """Estimate developmental age using confirmed / emerging / not_demonstrated band stages.

    v9 keeps one shared 3-stage scoring framework across all domains, but gives
    emerging performance stronger meaning in motor/postural subdomains because those
    skills often emerge gradually and inconsistently before stable mastery.
    """
    if not answers:
        return 6

    band_summary = summarize_answers_by_band(
        answers,
        min_yes_confirm=min_yes_confirm,
        yes_ratio_confirm=yes_ratio_confirm,
    )

    answered_months = sorted(band_summary.keys())
    confirmed_months = [m for m, info in band_summary.items() if info["stage"] == "confirmed"]

    if confirmed_months:
        return int(max(confirmed_months))

    emerging_months = [m for m, info in band_summary.items() if info["stage"] == "emerging"]
    if emerging_months:
        best_month = sorted(
            emerging_months,
            key=lambda m: (band_summary[m]["evidence_score"], m),
            reverse=True,
        )[0]
        return int(best_month)

    return int(min(answered_months))


In [ ]:
# ------------------------------------------------------------
# Q&A mode A — live parent
# ------------------------------------------------------------
def qna_agent_live(
    state: Dict[str, Any],
    category_key: str,
    min_yes_confirm: int = 2,
    yes_ratio_confirm: float = 0.60,
) -> Dict[str, Any]:
    """Run one domain interview in live mode using manual answers.

    v13 adds a lightweight follow-up only for ambiguous answers:
    - sometimes -> why only sometimes?
    - with_help -> what kind of help?
    - no -> what kind of "no"?

    These follow-up fields are stored for later review, but they do NOT overwrite
    milestone scoring in v13. Developmental scoring still uses the raw normalized answer.
    """
    print_banner(f"MILESTONE Q&A AGENT — {DOMAIN_CONFIG[category_key]['display']}")

    if "qna" not in state:
        state["qna"] = {}
    if "dev_age" not in state:
        state["dev_age"] = {}

    concern_profile = ensure_concern_profile(state)
    top_subdomains = [
        item["subdomain"]
        for item in concern_profile.get("top_subdomains", [])
        if SUBDOMAIN_TO_CATEGORY.get(item["subdomain"]) == category_key
    ]
    if top_subdomains:
        print("Concern-router emphasis in this domain:", ", ".join(top_subdomains[:3]))

    questions = build_milestone_questions(
        state,
        category_key,
        window_months=24,
        target_questions_per_band=3,
        max_bands=3,
        max_questions_total=9,
    )

    if not questions:
        raise ValueError(
            f"No milestone questions found for category_key={category_key}. "
            f"Check category mapping and CDC normalization."
        )

    asked = []
    questions_by_month = {}
    for q in questions:
        questions_by_month.setdefault(q["months"], []).append(q)

    for month in sorted(questions_by_month.keys()):
        print(f"\n--- Asking {DOMAIN_CONFIG[category_key]['display']} milestones around {month} months ---")
        for q in questions_by_month[month]:
            print(f"Subdomain: {q.get('subdomain', 'unspecified')} | selection_score: {q.get('selection_score', '')}")
            ans = input(q["question_text"] + "\n").strip()
            norm = normalize_answer(ans)

            followup_key = ""
            followup_label = ""
            if norm in FOLLOWUP_SCHEMAS:
                followup_key, followup_label = ask_live_followup(norm)

            asked.append({
                "question_id": q["question_id"],
                "months": q["months"],
                "milestone": q["milestone"],
                "subdomain": q.get("subdomain", "unspecified"),
                "raw_answer": ans,
                "norm_answer": norm,
                "score": ANSWER_SCORES[norm],
                "followup_key": followup_key,
                "followup_label": followup_label,
                "performance_barrier": infer_performance_barrier(norm, followup_key),
                "answer_status": "ok",
            })

    dev_age = compute_dev_age_from_answers(
        asked,
        min_yes_confirm=min_yes_confirm,
        yes_ratio_confirm=yes_ratio_confirm,
    )

    state["qna"][category_key] = asked
    state["dev_age"][category_key] = dev_age

    chrono = state["child"]["chronological_months"]

    print("\nQuestion / answer transcript:")
    for item in asked:
        extra = ""
        if item.get("followup_label"):
            extra = f" | followup: {item['followup_label']} | barrier: {item.get('performance_barrier', '')}"
        print(
            f"- [{item['months']}m | {item.get('subdomain', 'unspecified')}] "
            f"{item['milestone']} -> {item['norm_answer']}{extra}"
        )

    band_summary = summarize_answers_by_band(
        asked,
        min_yes_confirm=min_yes_confirm,
        yes_ratio_confirm=yes_ratio_confirm,
    )
    print("\nBand summary:")
    for month, info in band_summary.items():
        print(
            f"  {month}m | total={info['total']} | yes={info['yes']} | "
            f"partial={info['partial']} | no={info['no']} | yes_ratio={info['yes_ratio']} | "
            f"stage={info['stage']} | evidence={info['evidence_score']}"
            f"{' | motor_emphasis' if info.get('motor_emphasis') else ''}"
        )

    if category_key == "language_and_communication":
        lang_profile = compute_language_scoring_profile(state)
        effective_age = lang_profile.get('effective_dev_age_months', dev_age)
        print("\nLanguage scoring profile:")
        print(f"  raw language dev age: {lang_profile.get('raw_dev_age_months')}")
        print(f"  expressive/speech track: {lang_profile.get('track_dev_ages', {}).get('expressive_speech')}")
        print(f"  receptive track: {lang_profile.get('track_dev_ages', {}).get('receptive')}")
        print(f"  gesture track: {lang_profile.get('track_dev_ages', {}).get('gesture')}")
        print(f"  effective language dev age used for support planning: {effective_age}")
        print(
            f"\nEstimated developmental age for {DOMAIN_CONFIG[category_key]['display']}: "
            f"raw={dev_age} months | effective_planning_age={effective_age} months "
            f"(chronological age {chrono} months)"
        )
    else:
        print(
            f"\nEstimated developmental age for {DOMAIN_CONFIG[category_key]['display']}: "
            f"{dev_age} months (chronological age {chrono} months)"
        )

    return {
        "category": category_key,
        "dev_age_months": dev_age,
        "answers": asked,
        "band_summary": band_summary,
    }

In [ ]:
# ------------------------------------------------------------
# Q&A mode B — Gemini synthetic parent
# ------------------------------------------------------------

def gemini_parent_answer_one_question(
    child: Dict[str, Any],
    category_display: str,
    question_text: str,
    prior_answers: List[Dict[str, Any]],
    model: str = "gemini-2.5-flash",
    max_retries: int = 3,
) -> Dict[str, Any]:
    """Use Gemini to simulate one parent answer with retry/backoff protection.

    v13 keeps the v11 ability-focused prompting, and adds lightweight follow-up data
    for ambiguous answers. The follow-up is stored for review, but does NOT overwrite
    developmental scoring in v13.
    """
    if gemini_client is None:
        raise ValueError(
            "Gemini client is not available. Set GEMINI_API_KEY and rerun the environment setup cell."
        )

    if prior_answers:
        prior_text = "\n".join([f"- {x['question']} -> {x['answer']}" for x in prior_answers[-8:]])
    else:
        prior_text = "None yet."

    prompt = f"""
You are simulating a REAL parent answering developmental milestone questions about their child.

Rules:
- Answer ONLY from the child profile below.
- Be internally consistent across questions.
- Do NOT answer like a clinician or therapist.
- Answer like a parent who knows the child in daily life.
- Base the answer on the child's underlying ability when calm and given a fair chance, not only on behavior, rushing, refusal, distractibility, or frustration in the moment.
- Do NOT downgrade a skill just because the child is oppositional, impatient, rigid, or unwilling sometimes.
- Use "sometimes" when the skill itself is genuinely inconsistent or only partly there.
- Use "with_help" when the child truly needs physical help, setup help, or step-by-step support to do the skill.
- Use "no" when the child really cannot yet do the skill even with a fair opportunity.
- Choose ONLY one answer:
  yes
  sometimes
  with_help
  no
  not_sure

If you choose "sometimes", choose ONE matching followup_key from:
- not_consistent_yet
- distracted
- upset_or_refuses
- only_some_situations
- not_sure

If you choose "with_help", choose ONE matching followup_key from:
- physical_help
- reminders_prompting
- emotional_support
- showing_first
- not_sure

If you choose "no", choose ONE matching followup_key from:
- not_able_yet
- does_not_do_even_when_we_try
- upset_or_refuses
- distracted_before_doing
- not_sure_why

If you choose "yes" or "not_sure", return followup_key as an empty string.

Child profile:
- Name: {child.get('name')}
- Age: {child.get('chronological_months')} months
- Diagnosis / condition: {child.get('diagnosis')}
- Main concern: {child.get('concern')}

Current developmental category:
- {category_display}

Previous answers in this same interview:
{prior_text}

Question:
{question_text}

Return strict JSON only:
{{
  "answer": "yes|sometimes|with_help|no|not_sure",
  "followup_key": "",
  "reason": "one short parent-style reason that focuses on ability and may briefly mention a barrier if relevant"
}}
""".strip()

    last_error = None

    for attempt in range(max_retries):
        try:
            response = gemini_client.models.generate_content(
                model=model,
                contents=prompt,
            )

            raw_text = response.text if hasattr(response, "text") else str(response)
            parsed = extract_json_block(raw_text)

            answer = str(parsed.get("answer", "")).strip().lower().replace(" ", "_")
            if answer not in VALID_PARENT_ANSWERS:
                answer = "not_sure"

            followup_key = normalize_followup_answer(answer, str(parsed.get("followup_key", "")).strip())

            return {
                "answer": answer,
                "followup_key": followup_key,
                "followup_label": followup_label_from_key(answer, followup_key),
                "reason": str(parsed.get("reason", "")).strip() or raw_text[:200],
                "raw_text": raw_text,
                "status": "ok",
            }

        except Exception as e:
            last_error = e
            print(f"Gemini warning on question: {question_text}")
            print(f"Attempt {attempt+1}/{max_retries} failed: {e}")

            if attempt < max_retries - 1:
                time.sleep(10 * (attempt + 1))

    return {
        "answer": "not_sure",
        "followup_key": "",
        "followup_label": "",
        "reason": f"Fallback after Gemini failure: {last_error}",
        "raw_text": "",
        "status": "api_error",
    }

def qna_agent_gemini_parent(
    state: Dict[str, Any],
    category_key: str,
    ask_limit_per_band: int = 3,
    min_yes_confirm: int = 2,
    yes_ratio_confirm: float = 0.60,
    gemini_model: str = "gemini-2.5-flash",
    delay_between_questions_sec: int = 8,
) -> Dict[str, Any]:
    """Run one domain interview in synthetic-parent mode.

    API failures are marked as `api_error` and excluded from developmental-age scoring.
    v13 stores follow-up/barrier information, but developmental scoring still uses the
    raw normalized parent answer.
    """
    print_banner(f"MILESTONE Q&A AGENT — {DOMAIN_CONFIG[category_key]['display']}")

    if "qna" not in state:
        state["qna"] = {}
    if "dev_age" not in state:
        state["dev_age"] = {}

    concern_profile = ensure_concern_profile(state)
    top_subdomains = [
        item["subdomain"]
        for item in concern_profile.get("top_subdomains", [])
        if SUBDOMAIN_TO_CATEGORY.get(item["subdomain"]) == category_key
    ]
    if top_subdomains:
        print("Concern-router emphasis in this domain:", ", ".join(top_subdomains[:3]))

    questions = build_milestone_questions(
        state,
        category_key,
        window_months=24,
        target_questions_per_band=3,
        max_bands=3,
        max_questions_total=9,
    )

    if not questions:
        raise ValueError(
            f"No milestone questions found for category_key={category_key}. "
            f"Check category mapping and CDC normalization."
        )

    asked = []
    prior_answers_for_gemini = []

    questions_by_month = {}
    for q in questions:
        questions_by_month.setdefault(q["months"], []).append(q)

    for month in sorted(questions_by_month.keys()):
        print(f"\n--- Asking {DOMAIN_CONFIG[category_key]['display']} milestones around {month} months ---")

        for q in questions_by_month[month][:ask_limit_per_band]:
            print(f"Subdomain: {q.get('subdomain', 'unspecified')} | selection_score: {q.get('selection_score', '')}")
            sim = gemini_parent_answer_one_question(
                child=state["child"],
                category_display=DOMAIN_CONFIG[category_key]["display"],
                question_text=q["question_text"],
                prior_answers=prior_answers_for_gemini,
                model=gemini_model,
                max_retries=3,
            )

            norm = normalize_answer(sim["answer"])
            followup_key = normalize_followup_answer(norm, sim.get("followup_key", ""))
            followup_label = followup_label_from_key(norm, followup_key)

            asked_item = {
                "question_id": q["question_id"],
                "months": q["months"],
                "milestone": q["milestone"],
                "subdomain": q.get("subdomain", "unspecified"),
                "raw_answer": sim["answer"],
                "norm_answer": norm,
                "score": ANSWER_SCORES[norm],
                "followup_key": followup_key,
                "followup_label": followup_label,
                "performance_barrier": infer_performance_barrier(norm, followup_key),
                "parent_reason": sim["reason"],
                "answer_status": sim.get("status", "ok"),
            }
            asked.append(asked_item)

            prior_answers_for_gemini.append({
                "question": q["question_text"],
                "answer": sim["answer"] + (f" | {followup_label}" if followup_label else ""),
            })

            extra = ""
            if followup_label:
                extra = f" | followup: {followup_label} | barrier: {asked_item['performance_barrier']}"

            print(
                f"Q: {q['question_text']}\n"
                f"A (Gemini-parent): {sim['answer']}{extra} | "
                f"status: {sim.get('status', 'ok')} | "
                f"reason: {sim['reason']}"
            )

            time.sleep(delay_between_questions_sec)

    usable_answers = [a for a in asked if a.get("answer_status") != "api_error"]

    dev_age = compute_dev_age_from_answers(
        usable_answers,
        min_yes_confirm=min_yes_confirm,
        yes_ratio_confirm=yes_ratio_confirm,
    )

    state["qna"][category_key] = asked
    state["dev_age"][category_key] = dev_age

    chrono = state["child"]["chronological_months"]

    print("\nQuestion / answer transcript:")
    for item in asked:
        extra = ""
        if item.get("followup_label"):
            extra = f" | followup: {item['followup_label']} | barrier: {item.get('performance_barrier', '')}"
        print(
            f"- [{item['months']}m | {item.get('subdomain', 'unspecified')}] {item['milestone']} -> {item['norm_answer']}"
            f"{extra} | status: {item.get('answer_status', 'ok')} | reason: {item.get('parent_reason', '')}"
        )

    band_summary = summarize_answers_by_band(
        usable_answers,
        min_yes_confirm=min_yes_confirm,
        yes_ratio_confirm=yes_ratio_confirm,
    )
    print("\nBand summary:")
    for month, info in band_summary.items():
        print(
            f"  {month}m | total={info['total']} | yes={info['yes']} | "
            f"partial={info['partial']} | no={info['no']} | yes_ratio={info['yes_ratio']} | "
            f"stage={info['stage']} | evidence={info['evidence_score']}"
            f"{' | motor_emphasis' if info.get('motor_emphasis') else ''}"
        )

    if category_key == "language_and_communication":
        lang_profile = compute_language_scoring_profile(state)
        effective_age = lang_profile.get('effective_dev_age_months', dev_age)
        print("\nLanguage scoring profile:")
        print(f"  raw language dev age: {lang_profile.get('raw_dev_age_months')}")
        print(f"  expressive/speech track: {lang_profile.get('track_dev_ages', {}).get('expressive_speech')}")
        print(f"  receptive track: {lang_profile.get('track_dev_ages', {}).get('receptive')}")
        print(f"  gesture track: {lang_profile.get('track_dev_ages', {}).get('gesture')}")
        print(f"  effective language dev age used for support planning: {effective_age}")
        print(
            f"\nEstimated developmental age for {DOMAIN_CONFIG[category_key]['display']}: "
            f"raw={dev_age} months | effective_planning_age={effective_age} months "
            f"(chronological age {chrono} months)"
        )
    else:
        print(
            f"\nEstimated developmental age for {DOMAIN_CONFIG[category_key]['display']}: "
            f"{dev_age} months (chronological age {chrono} months)"
        )

    return {
        "category": category_key,
        "dev_age_months": dev_age,
        "answers": asked,
        "band_summary": band_summary,
    }

In [ ]:

# ------------------------------------------------------------
# Support tier + shared planning engine
# ------------------------------------------------------------


def get_category_concern_peak(state: Dict[str, Any], category_key: str) -> float:
    concern_profile = ensure_concern_profile(state)
    subdomains = CATEGORY_TO_SUBDOMAINS.get(category_key, [])
    if not subdomains:
        return 0.0
    return max(float(concern_profile["subdomain_weights"].get(s, 0.0)) for s in subdomains)

LANGUAGE_SCORING_TRACKS = {
    "expressive_speech": {
        "subdomains": {
            "expressive_language",
            "speech_intelligibility",
            "early_vocalization_and_babbling",
            "conversation_narrative",
        }
    },
    "receptive": {
        "subdomains": {"receptive_language"}
    },
    "gesture": {
        "subdomains": {"gestural_communication"}
    },
}

def _answers_for_subdomains(
    answers: List[Dict[str, Any]],
    subdomains: set,
) -> List[Dict[str, Any]]:
    return [
        a for a in answers
        if str(a.get("subdomain", "unspecified")) in subdomains
    ]

def compute_language_scoring_profile(
    state: Dict[str, Any],
    min_yes_confirm: int = 2,
    yes_ratio_confirm: float = 0.60,
) -> Dict[str, Any]:
    """Compute an internal split scoring profile for Language / Communication.

    v9 keeps the public domain label the same, but internally separates:
    - expressive/speech-related items
    - receptive language items
    - gestural communication items

    The goal is to reduce a common failure mode where strong gestures or strong
    comprehension mask a more significant expressive/speech weakness. Version 9
    keeps this split and combines it with the new confirmed / emerging / not_demonstrated
    framework used elsewhere in the notebook.
    """
    raw_dev_age = state.get("dev_age", {}).get("language_and_communication")
    qna_answers = [
        a for a in state.get("qna", {}).get("language_and_communication", [])
        if a.get("answer_status", "ok") != "api_error"
    ]

    if raw_dev_age is None or not qna_answers:
        return {
            "raw_dev_age_months": raw_dev_age,
            "effective_dev_age_months": raw_dev_age,
            "track_dev_ages": {},
            "track_weights": {},
            "track_counts": {},
        }

    track_dev_ages = {}
    track_counts = {}
    for track_name, cfg in LANGUAGE_SCORING_TRACKS.items():
        subset = _answers_for_subdomains(qna_answers, cfg["subdomains"])
        track_counts[track_name] = len(subset)
        if subset:
            track_dev_ages[track_name] = compute_dev_age_from_answers(
                subset,
                min_yes_confirm=min_yes_confirm,
                yes_ratio_confirm=yes_ratio_confirm,
            )

    concern_profile = ensure_concern_profile(state)
    sub_w = concern_profile.get("subdomain_weights", {})

    expressive_pressure = max(
        float(sub_w.get("expressive_language", 0.0)),
        float(sub_w.get("speech_intelligibility", 0.0)),
        float(sub_w.get("early_vocalization_and_babbling", 0.0)),
        float(sub_w.get("conversation_narrative", 0.0)),
    )
    receptive_pressure = float(sub_w.get("receptive_language", 0.0))
    gesture_pressure = float(sub_w.get("gestural_communication", 0.0))

    # Gesture is useful communication information, but v9 caps its influence so it does not
    # fully mask expressive/speech weakness.
    track_weights = {}
    if track_counts.get("expressive_speech", 0) > 0:
        track_weights["expressive_speech"] = max(0.35, expressive_pressure)
    if track_counts.get("receptive", 0) > 0:
        track_weights["receptive"] = max(0.15, receptive_pressure)
    if track_counts.get("gesture", 0) > 0:
        track_weights["gesture"] = min(0.35, max(0.05, gesture_pressure * 0.5))

    if not track_weights:
        effective_age = raw_dev_age
    else:
        weighted_total = 0.0
        weight_sum = 0.0
        for track_name, weight in track_weights.items():
            age = track_dev_ages.get(track_name, raw_dev_age)
            weighted_total += float(age) * float(weight)
            weight_sum += float(weight)

        weighted_age = round(weighted_total / max(weight_sum, 1e-9))
        effective_age = min(int(raw_dev_age), int(weighted_age))

        # When expressive/speech concern is clearly dominant, do not let strong gesture or
        # receptive performance pull the language age above the expressive/speech track.
        if expressive_pressure >= 0.50 and "expressive_speech" in track_dev_ages:
            effective_age = min(effective_age, int(track_dev_ages["expressive_speech"]))

    return {
        "raw_dev_age_months": int(raw_dev_age),
        "effective_dev_age_months": int(effective_age),
        "track_dev_ages": track_dev_ages,
        "track_weights": {k: round(v, 2) for k, v in track_weights.items()},
        "track_counts": track_counts,
    }

def get_effective_dev_age(state: Dict[str, Any], category_key: str) -> Optional[int]:
    raw = state.get("dev_age", {}).get(category_key)
    if raw is None:
        return None
    if category_key != "language_and_communication":
        return int(raw)

    profile = compute_language_scoring_profile(state)
    eff = profile.get("effective_dev_age_months", raw)
    return int(eff) if eff is not None else int(raw)

def compute_support_metrics(state: Dict[str, Any], category_key: str) -> Dict[str, Any]:
    """Compute the continuous support score used for support-tier assignment.

    v9 keeps this deterministic:
    - milestone gap is still the strongest signal
    - starting delay remains meaningful
    - concern pressure is present but deliberately weaker
    - language support now uses an *effective* within-domain dev age so expressive/speech
      weakness is not masked by stronger gesture or receptive performance
    """
    child = state["child"]
    concern_profile = ensure_concern_profile(state)

    chrono = min(child["chronological_months"], 60)
    raw_dev_age = state["dev_age"].get(category_key, chrono)
    effective_dev_age = get_effective_dev_age(state, category_key)
    if effective_dev_age is None:
        effective_dev_age = raw_dev_age

    delay_est = state["delay_estimates"].get(category_key, {}).get("delay_months", 0)
    concern_domain_weight = float(concern_profile["domain_weights"].get(category_key, 0.0))
    concern_subdomain_peak = float(get_category_concern_peak(state, category_key))

    gap = max(0, chrono - int(effective_dev_age))

    light_gap_threshold = max(2, round(0.10 * chrono))
    primary_gap_threshold = max(5, round(0.20 * chrono))

    light_delay_threshold = max(4, round(0.10 * chrono))
    primary_delay_threshold = max(6, round(0.20 * chrono))

    gap_component = min(1.5, gap / max(primary_gap_threshold, 1))
    delay_component = min(1.5, delay_est / max(primary_delay_threshold, 1))
    concern_component = max(concern_domain_weight, concern_subdomain_peak * 0.85)

    support_score = (
        0.55 * gap_component +
        0.25 * delay_component +
        0.20 * concern_component
    )

    if gap >= primary_gap_threshold or support_score >= 0.95:
        tier = "needs_special_support"
    elif (
        support_score >= 0.42
        or (gap >= light_gap_threshold and concern_component >= 0.35)
        or (delay_est >= light_delay_threshold and concern_component >= 0.35)
    ):
        tier = "monitor_and_enrich"
    else:
        tier = "no_special_support"

    result = {
        "chronological_months": chrono,
        "raw_dev_age_months": int(raw_dev_age) if raw_dev_age is not None else None,
        "effective_dev_age_months": int(effective_dev_age) if effective_dev_age is not None else None,
        "estimated_dev_age_months": int(effective_dev_age) if effective_dev_age is not None else None,
        "milestone_gap_months": gap,
        "estimated_delay_months": delay_est,
        "concern_domain_weight": concern_domain_weight,
        "concern_subdomain_peak": concern_subdomain_peak,
        "light_gap_threshold": light_gap_threshold,
        "primary_gap_threshold": primary_gap_threshold,
        "light_delay_threshold": light_delay_threshold,
        "primary_delay_threshold": primary_delay_threshold,
        "support_score": round(float(support_score), 3),
        "tier": tier,
    }

    if category_key == "language_and_communication":
        result["language_scoring_profile"] = compute_language_scoring_profile(state)

    return result

def get_support_tier(state: Dict[str, Any], category_key: str) -> str:
    return compute_support_metrics(state, category_key)["tier"]

def no_special_support_needed(state: Dict[str, Any], category_key: str) -> bool:
    return get_support_tier(state, category_key) == "no_special_support"


def determine_family_guidance_floor(state: Dict[str, Any]) -> Dict[str, Any]:
    """Create a soft planning floor when all categories are technically no special support.

    This does NOT change the underlying support tier. It only ensures the family still
    receives short age-appropriate enrichment-and-observation activities in the category
    most aligned with the expressed concern.
    """
    concern_profile = ensure_concern_profile(state)
    child = state["child"]

    supported = [k for k in DOMAIN_CONFIG if get_support_tier(state, k) != "no_special_support"]
    if supported:
        info = {
            "enabled": False,
            "mode": None,
            "category_key": None,
            "planning_tier": None,
            "target_weekly_minutes": 0,
            "summary": "",
        }
        state["family_guidance_floor"] = info
        return info

    ranked = sorted(
        DOMAIN_CONFIG.keys(),
        key=lambda k: (
            float(concern_profile.get("domain_weights", {}).get(k, 0.0)),
            float(get_category_concern_peak(state, k)),
            float(state.get("delay_estimates", {}).get(k, {}).get("delay_months", 0)),
        ),
        reverse=True,
    )
    category_key = ranked[0] if ranked else "language_and_communication"
    category_display = DOMAIN_CONFIG[category_key]["display"]
    daily_time_min = int(child.get("daily_time_min", 10))
    target_weekly_minutes = min(max(15, daily_time_min * 3), daily_time_min * 5)

    summary = (
        f"Based on the milestone interview, {child['name']} does not currently appear to need scheduled special support. "
        f"Because the family expressed concern about {category_display}, the system will still provide short age-appropriate "
        f"enrich-and-observe activities in this category."
    )

    info = {
        "enabled": True,
        "mode": "enrich_and_observe",
        "category_key": category_key,
        "category_display": category_display,
        "planning_tier": "enrich_and_observe",
        "target_weekly_minutes": int(target_weekly_minutes),
        "summary": summary,
    }
    state["family_guidance_floor"] = info
    return info


def is_family_guidance_category(state: Dict[str, Any], category_key: str) -> bool:
    floor = state.get("family_guidance_floor", {})
    return bool(floor.get("enabled") and floor.get("category_key") == category_key)

def select_next_milestones(state: Dict[str, Any], category_key: str, max_milestones: int = 6) -> Dict[str, Any]:
    """Select milestones either for support planning or for soft enrich-and-observe guidance."""
    child = state["child"]
    dev_age = get_effective_dev_age(state, category_key)
    soft_floor_active = is_family_guidance_category(state, category_key)

    if dev_age is None and not soft_floor_active:
        raise ValueError(f"No developmental age found for {category_key}. Run Q&A first.")

    if no_special_support_needed(state, category_key) and not soft_floor_active:
        return {
            "status": "no_special_support",
            "message": f"We do not think {child['name']} has a meaningful delay in {DOMAIN_CONFIG[category_key]['display']} and may not need special support in this category right now.",
            "milestones": [],
        }

    chrono = min(child["chronological_months"], 60)
    if soft_floor_active:
        min_m = max(2, chrono - 3)
        max_m = min(60, chrono + 3)
    else:
        min_m = max(2, dev_age)
        max_m = min(60, dev_age + 12)

    subset = get_category_questions(category_key, min_months=min_m, max_months=max_m)
    if subset.empty:
        return {"status": "success", "milestones": [], "mode": "soft_floor" if soft_floor_active else "support"}

    subset = subset.sort_values(["months", "milestone"]).copy()
    milestones = []
    seen = set()

    for _, row in subset.iterrows():
        key = (int(row["months"]), str(row["milestone"]).strip())
        if key in seen:
            continue
        seen.add(key)
        milestones.append({
            "months": int(row["months"]),
            "milestone": row["milestone"],
            "subdomain": str(row.get("subdomain", "unspecified")),
        })

    return {
        "status": "success",
        "milestones": milestones[:max_milestones],
        "mode": "soft_floor" if soft_floor_active else "support",
    }

def get_category_activity_guardrails(category_key: str) -> str:
    """Return domain-specific activity guardrails to reduce cross-domain drift."""
    if category_key == "movement_and_physical":
        return """
Category-specific rules for Movement / Physical:
- Focus on posture, strength, balance, coordination, reaching, grasping, sitting, crawling, standing, walking, or fine-motor use.
- The main goal should be physical skill practice.
- Do NOT make this mainly about speech, naming, imitation of sounds, or social-emotional labeling.
"""

    if category_key == "language_and_communication":
        return """
Category-specific rules for Language / Communication:
- Focus on sounds, babbling, turn-taking vocalization, following simple verbal directions, gestures for communication, word use, imitation of sounds, naming, requesting, commenting, comprehension, or speech clarity.
- The main goal should be communication.
- Do NOT make this mainly about gross motor practice, reaching, balance, climbing, or general physical exercise.
- If an object is used, it should support communication, not be the main task.
"""

    if category_key == "social_and_emotional":
        return """
Category-specific rules for Social / Emotional:
- Focus on eye contact, shared attention, social reciprocity, imitation of facial expressions, joint play, turn-taking, response to name, emotional connection, emotional regulation, separation, or interaction with caregiver or peers.
- The main goal should be social engagement or emotional regulation.
- Do NOT make this mainly about motor strengthening or naming/labeling language tasks unless the social interaction is clearly primary.
"""

    if category_key == "cognitive":
        return """
Category-specific rules for Cognitive / Adaptive:
- Focus on attention, problem-solving, object permanence, simple cause-and-effect, routines, imitation of actions, functional play, early self-help, adaptive understanding, or following directions.
- The main goal should be cognitive/adaptive skill building.
- Do NOT make this mainly about speech production or gross motor strengthening unless the cognitive/adaptive task is clearly primary.
"""

    return ""

def generate_category_activity_bank(
    state: Dict[str, Any],
    category_key: str,
    model: str = "gpt-4o-mini",
    activities_per_category: int = 6,
) -> Dict[str, Any]:
    """Generate one activity bank per category."""
    child = state["child"]
    category_display = DOMAIN_CONFIG[category_key]["display"]
    support_tier = get_support_tier(state, category_key)
    soft_floor_active = is_family_guidance_category(state, category_key)
    planning_tier = "enrich_and_observe" if soft_floor_active else support_tier
    support_metrics = compute_support_metrics(state, category_key)
    safety_profile = ensure_safety_profile(state)
    safety_constraints_block = format_safety_constraints_for_prompt(safety_profile)
    next_steps = select_next_milestones(state, category_key)

    if next_steps["status"] == "no_special_support":
        state["activity_banks"][category_key] = {
            "status": "no_special_support",
            "support_tier": support_tier,
            "planning_tier": planning_tier,
            "summary": next_steps["message"],
            "activities": [],
        }
        return state["activity_banks"][category_key]

    chrono_months = min(child["chronological_months"], 60)
    dev_age = chrono_months if soft_floor_active else state["dev_age"][category_key]
    milestone_gap = max(0, chrono_months - dev_age)
    category_guardrails = get_category_activity_guardrails(category_key)
    soft_floor_block = ""
    if soft_floor_active:
        soft_floor_block = (
            "This category is being generated under the family guidance floor. "
            "Use age-appropriate, low-intensity enrichment and observation activities. "
            "Do not imply therapy-level intensity or a significant delay. "
            "The goal is support, confidence-building, and structured observation."
        )

    milestone_lines = "\n".join(
        [f"- ({m['months']} months | {m.get('subdomain', 'unspecified')}) {m['milestone']}" for m in next_steps["milestones"]]
    )
    if not milestone_lines:
        milestone_lines = "- No specific milestone items available in this range."

    if openai_client is None:
        fallback_activities = []
        for i, m in enumerate(next_steps["milestones"][:activities_per_category], start=1):
            fallback_activities.append({
                "activity_id": f"{category_key}_{i}",
                "title": f"Practice: {m['milestone']}",
                "instructions": f"Use a short, play-based home activity to practice: {m['milestone']}.",
                "duration_min": 5,
                "materials": "common toys or household items",
                "level": "current_or_next",
                "goal": planning_tier,
                "category": category_display,
                "is_extended_activity": False,
                "extended_reason": "",
            })

        while len(fallback_activities) < activities_per_category:
            i = len(fallback_activities) + 1
            fallback_activities.append({
                "activity_id": f"{category_key}_{i}",
                "title": f"{category_display} practice {i}",
                "instructions": f"Do one short parent-guided home activity that supports {category_display.lower()} skills.",
                "duration_min": 5,
                "materials": "common toys or household items",
                "level": "current_or_next",
                "goal": planning_tier,
                "category": category_display,
                "is_extended_activity": False,
                "extended_reason": "",
            })

        fallback_activities = apply_safety_constraints_to_activities(state, category_key, fallback_activities)
        state["activity_banks"][category_key] = {
            "status": "fallback",
            "support_tier": support_tier,
            "planning_tier": planning_tier,
            "summary": f"Fallback activity bank used for {category_display}.",
            "activities": fallback_activities,
        }
        return state["activity_banks"][category_key]

    prompt = f"""
You are a pediatric home-support planning agent helping a parent at home.

This is NOT a diagnosis and NOT a formal treatment plan.
Create a CATEGORY ACTIVITY BANK, not a day-by-day schedule.

Child:
- Name: {child['name']}
- Chronological age: {child['chronological_months']} months
- Diagnosis / condition: {child['diagnosis']}
- Parent concern: {child['concern']}
- Category: {category_display}
- Support tier for this category: {planning_tier}
- Estimated developmental age in this category: {dev_age} months
- Estimated milestone gap in this category: {milestone_gap} months\n- Continuous support score for this category: {support_metrics['support_score']}\n
Relevant milestone targets:
{milestone_lines}

Category-specific guardrails:\n{category_guardrails}\n\nSafety / practical constraints inferred from diagnosis + concern:\n{safety_constraints_block}\n\nThese safety items are real planning constraints, not optional suggestions. Do not propose activities that violate the hard-avoid list. Prefer the safer adaptations when relevant.\n\nTask:\nCreate {activities_per_category} realistic home activities for this category.

Instructions:
1. Activities must fit the child's chronological age and estimated developmental level.
2. Activities should be practical for home.
3. Keep language parent-friendly.
4. Include a mix of current-level practice and near next-step practice.
5. Most activities should be short, repeatable, and realistic for home: usually 3, 5, 7, or 10 minutes.
6. The CORE weekly schedule is for short at-home activities only.
7. Context-dependent activities such as playdates, park meetups, playground peer practice, group social activities, or community participation must NEVER be written as normal daily home activities.
8. If you include a playdate-, park-, playground-, meetup-, or group-type activity, mark it as an extended activity intended for OPTIONAL WEEKLY BONUS use only.
9. Any context-dependent bonus activity should be 30, 35, 40, or 45 minutes, not 5 or 10 minutes.
10. Do not make more than 1 context-dependent bonus activity in this category.
11. Each activity must clearly belong to THIS category first and foremost.
12. Avoid cross-domain drift. The main goal of the activity should be obvious from the category.
13. If an activity could fit multiple domains, choose the version that best matches this category's core skill.
14. Set "goal" to exactly the planning tier for this category: {planning_tier}
15. Return strict JSON only.

Required JSON:
{{
  "summary": "...",
  "activities": [
    {{
      "activity_id": "1",
      "title": "...",
      "instructions": "...",
      "duration_min": 5,
      "materials": "...",
      "level": "current_or_next",
      "goal": "{planning_tier}",
      "category": "<category name>",
      "is_extended_activity": false,
      "extended_reason": ""
    }}
  ]
}}
""".strip()

    try:
        resp = openai_client.chat.completions.create(
            model=model,
            temperature=0.3,
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": "You return strict JSON only and stay non-diagnostic, practical, and parent-friendly."},
                {"role": "user", "content": prompt},
            ],
        )

        bank = json.loads(resp.choices[0].message.content)

        if "activities" not in bank or not isinstance(bank["activities"], list):
            bank["activities"] = []

        for idx, activity in enumerate(bank["activities"], start=1):
            activity["activity_id"] = activity.get("activity_id", f"{category_key}_{idx}")
            activity["category"] = activity.get("category", category_display)
            activity["duration_min"] = activity.get("duration_min", 5)
            activity["materials"] = activity.get("materials", "common household items")
            activity["level"] = activity.get("level", "current_or_next")
            activity["goal"] = planning_tier
            activity["is_extended_activity"] = activity.get("is_extended_activity", False)
            activity["extended_reason"] = activity.get("extended_reason", "")

        bank["activities"] = apply_safety_constraints_to_activities(state, category_key, bank["activities"][:activities_per_category])

        state["activity_banks"][category_key] = {
            "status": "success",
            "support_tier": support_tier,
            "planning_tier": planning_tier,
            "summary": bank.get("summary", f"Created activity bank for {category_display}."),
            "activities": bank["activities"],
        }
        return state["activity_banks"][category_key]

    except Exception as e:
        fallback_activities = []
        for i, m in enumerate(next_steps["milestones"][:activities_per_category], start=1):
            fallback_activities.append({
                "activity_id": f"{category_key}_{i}",
                "title": f"Practice: {m['milestone']}",
                "instructions": f"Use a short, play-based home activity to practice: {m['milestone']}.",
                "duration_min": 5,
                "materials": "common toys or household items",
                "level": "current_or_next",
                "goal": planning_tier,
                "category": category_display,
                "is_extended_activity": False,
                "extended_reason": "",
            })

        while len(fallback_activities) < activities_per_category:
            i = len(fallback_activities) + 1
            fallback_activities.append({
                "activity_id": f"{category_key}_{i}",
                "title": f"{category_display} practice {i}",
                "instructions": f"Do one short parent-guided home activity that supports {category_display.lower()} skills.",
                "duration_min": 5,
                "materials": "common toys or household items",
                "level": "current_or_next",
                "goal": planning_tier,
                "category": category_display,
                "is_extended_activity": False,
                "extended_reason": "",
            })

        fallback_activities = apply_safety_constraints_to_activities(state, category_key, fallback_activities)
        state["activity_banks"][category_key] = {
            "status": "fallback",
            "support_tier": support_tier,
            "planning_tier": planning_tier,
            "summary": f"Fallback activity bank used for {category_display} because OpenAI failed: {e}",
            "activities": fallback_activities,
        }
        return state["activity_banks"][category_key]


CONTEXT_DEPENDENT_BONUS_PATTERN = re.compile(
    r"\b(playdate|park meetup|playground peer|group social|group activity|peer practice|community class|outing|meetup)\b",
    flags=re.IGNORECASE,
)

def is_context_dependent_bonus_activity(activity: Dict[str, Any]) -> bool:
    text = " ".join(
        [
            str(activity.get("title", "")),
            str(activity.get("instructions", "")),
            str(activity.get("materials", "")),
            str(activity.get("extended_reason", "")),
        ]
    )
    return bool(CONTEXT_DEPENDENT_BONUS_PATTERN.search(text))

def apply_safety_constraints_to_activities(
    state: Dict[str, Any],
    category_key: str,
    activities: List[Dict[str, Any]],
) -> List[Dict[str, Any]]:
    """Deterministically adapt or sanitize activities based on the broad safety profile.

    This is intentionally broad and risk-based rather than diagnosis-name-specific.
    """
    profile = ensure_safety_profile(state)
    risk_scores = profile.get("risk_scores", {})

    high_fall_or_mobility = (
        risk_scores.get("falls_balance_gait", 0.0) >= 0.35
        or risk_scores.get("mobility_equipment_support", 0.0) >= 0.35
    )
    low_tone_or_fatigue = risk_scores.get("postural_low_tone_fatigue", 0.0) >= 0.35
    feeding_risk = risk_scores.get("feeding_or_oral_motor", 0.0) >= 0.35
    regulation_risk = risk_scores.get("regulation_frustration", 0.0) >= 0.35
    sensory_risk = risk_scores.get("sensory_sensitivity", 0.0) >= 0.35

    safe_activities = []

    for activity in activities:
        a = dict(activity)
        title = str(a.get("title", ""))
        instructions = str(a.get("instructions", ""))
        materials = str(a.get("materials", "common household items"))
        duration_min = int(a.get("duration_min", 5))
        lower_text = f"{title} {instructions}".lower()

        if high_fall_or_mobility and re.search(r'\b(jump|jumping|trampoline|hop|hopping|climb|climbing|playground)\b', lower_text):
            a["title"] = "Supported Balance and Reach Practice"
            a["instructions"] = (
                "Use a ground-level activity with close adult support. Practice supported standing, "
                "seated or supported reaching, and safe weight shifts while keeping both feet on a stable surface. "
                "Avoid jumping, climbing, or unstable surfaces."
            )
            a["materials"] = "stable chair or couch, wall or caregiver support, favorite toys"
            a["duration_min"] = min(duration_min, 7)
            title = a["title"]
            instructions = a["instructions"]
            materials = a["materials"]
            duration_min = a["duration_min"]
            lower_text = f"{title} {instructions}".lower()

        if high_fall_or_mobility and category_key == "movement_and_physical":
            if "close adult support" not in instructions.lower() and "stable support" not in instructions.lower():
                a["instructions"] = instructions.rstrip() + " Use stable support and close adult supervision throughout."
                instructions = a["instructions"]

        if low_tone_or_fatigue:
            a["duration_min"] = min(int(a.get("duration_min", duration_min)), 7)
            if "rest break" not in instructions.lower():
                a["instructions"] = instructions.rstrip() + " Keep the activity in short bouts and pause for rest if the child tires."
                instructions = a["instructions"]

        if regulation_risk:
            a["duration_min"] = min(int(a.get("duration_min", duration_min)), 5)
            if "predictable" not in instructions.lower():
                a["instructions"] = instructions.rstrip() + " Keep it short and predictable, offer a simple choice, and stop before frustration escalates."
                instructions = a["instructions"]

        if sensory_risk and "lower-noise" not in instructions.lower():
            a["instructions"] = instructions.rstrip() + " Use lower-noise, lower-clutter materials when possible."
            instructions = a["instructions"]

        if feeding_risk and re.search(r'\b(food|snack|chew|mouth|eat|drink|feeding)\b', lower_text):
            if "upright" not in instructions.lower():
                a["instructions"] = instructions.rstrip() + " Keep the child upright and closely supervised. Avoid choking-risk foods and stop if coughing, gagging, or distress occurs."
                instructions = a["instructions"]

        if is_context_dependent_bonus_activity(a):
            a["is_extended_activity"] = True
            normalized_duration = int(a.get("duration_min", duration_min))
            if normalized_duration < 30:
                normalized_duration = 30
            if normalized_duration > 45:
                normalized_duration = 45
            a["duration_min"] = normalized_duration
            if "optional weekly bonus" not in str(a.get("extended_reason", "")).lower():
                a["extended_reason"] = (
                    "Longer, context-dependent social/community activity best treated as an optional weekly bonus rather than a short daily at-home task."
                )

        safe_activities.append(a)

    return safe_activities

def allocate_weekly_slots(state: Dict[str, Any]) -> Dict[str, Any]:
    child = state["child"]

    if "daily_time_min" not in state["child"]:
        raise ValueError(
            "Missing daily_time_min in state['child']. "
            "Check intake_agent_live() and make sure it stores daily_time_min."
        )

    chrono = min(child["chronological_months"], 60)
    daily_time_min = int(child["daily_time_min"])
    weekly_minutes = daily_time_min * 5

    supported_categories = []
    gap_by_category = {}
    weight_by_category = {}

    for category_key in DOMAIN_CONFIG:
        tier = get_support_tier(state, category_key)
        if tier == "no_special_support":
            continue

        supported_categories.append(category_key)

        dev_age = get_effective_dev_age(state, category_key)
        if dev_age is None:
            dev_age = state["dev_age"].get(category_key, chrono)
        gap = max(0, chrono - dev_age)
        gap_by_category[category_key] = gap

        if tier == "needs_special_support":
            weight_by_category[category_key] = max(1, gap)
        else:  # monitor_and_enrich
            weight_by_category[category_key] = max(1, gap) * 0.5

    soft_floor = determine_family_guidance_floor(state)
    if not supported_categories and soft_floor.get("enabled"):
        category_key = soft_floor["category_key"]
        target_minutes = min(int(soft_floor.get("target_weekly_minutes", 20)), weekly_minutes)
        allocation = {
            "daily_time_min": daily_time_min,
            "weekly_minutes": weekly_minutes,
            "supported_categories": [category_key],
            "gap_by_category": {category_key: 0},
            "target_minutes_by_category": {category_key: target_minutes},
            "planning_mode": "family_guidance_floor",
        }
        state["weekly_slot_allocation"] = allocation
        return allocation

    if not supported_categories:
        allocation = {
            "daily_time_min": daily_time_min,
            "weekly_minutes": weekly_minutes,
            "supported_categories": [],
            "gap_by_category": {},
            "target_minutes_by_category": {},
            "planning_mode": "none",
        }
        state["weekly_slot_allocation"] = allocation
        return allocation

    base_minutes_per_category = max(5, daily_time_min // 2)
    target_minutes_by_category = {k: base_minutes_per_category for k in supported_categories}

    used_minutes = base_minutes_per_category * len(supported_categories)
    remaining_minutes = max(0, weekly_minutes - used_minutes)

    weights = weight_by_category.copy()
    total_weight = sum(weights.values())

    if total_weight > 0 and remaining_minutes > 0:
        for k in supported_categories:
            extra = round(remaining_minutes * (weights[k] / total_weight))
            target_minutes_by_category[k] += extra

    total_target = sum(target_minutes_by_category.values())
    while total_target > weekly_minutes:
        biggest = max(target_minutes_by_category, key=target_minutes_by_category.get)
        if target_minutes_by_category[biggest] > 5:
            target_minutes_by_category[biggest] -= 1
            total_target -= 1
        else:
            break

    allocation = {
        "daily_time_min": daily_time_min,
        "weekly_minutes": weekly_minutes,
        "supported_categories": supported_categories,
        "gap_by_category": gap_by_category,
        "target_minutes_by_category": target_minutes_by_category,
        "planning_mode": "tiered_support",
    }

    state["weekly_slot_allocation"] = allocation
    return allocation

def pick_activity_that_fits(
    activities: List[Dict[str, Any]],
    used_indices: set,
    remaining_minutes: int,
) -> Optional[Dict[str, Any]]:
    candidates = []

    for idx, activity in enumerate(activities):
        if idx in used_indices:
            continue
        if activity.get("is_extended_activity", False):
            continue
        if is_context_dependent_bonus_activity(activity):
            continue
        duration = int(activity.get("duration_min", 5))
        if duration <= remaining_minutes:
            candidates.append((duration, idx, activity))

    if not candidates:
        return None

    candidates = sorted(candidates, key=lambda x: x[0])
    duration, idx, activity = candidates[0]
    used_indices.add(idx)
    return activity

def pick_weekly_bonus_extended_activity(
    state: Dict[str, Any],
    extended_in_schedule_threshold: int = 15,
    bonus_extended_min_min: int = 30,
    bonus_extended_cap_min: int = 45,
) -> Optional[Dict[str, Any]]:
    daily_time_min = int(state["child"]["daily_time_min"])

    if daily_time_min >= extended_in_schedule_threshold:
        return None

    allocation = state.get("weekly_slot_allocation", {})
    gap_by_category = allocation.get("gap_by_category", {})

    candidate_categories = sorted(
        gap_by_category.keys(),
        key=lambda k: gap_by_category[k],
        reverse=True,
    )

    for category_key in candidate_categories:
        bank = state.get("activity_banks", {}).get(category_key, {})
        activities = bank.get("activities", [])

        for activity in activities:
            if not activity.get("is_extended_activity", False):
                continue

            duration = int(activity.get("duration_min", 5))
            if is_context_dependent_bonus_activity(activity):
                duration = max(duration, bonus_extended_min_min)
            if duration < bonus_extended_min_min and is_context_dependent_bonus_activity(activity):
                continue
            if duration > bonus_extended_cap_min:
                continue

            return {
                "category_key": category_key,
                "category": DOMAIN_CONFIG[category_key]["display"],
                "title": activity.get("title"),
                "instructions": activity.get("instructions"),
                "duration_min": duration,
                "materials": activity.get("materials", "common household items"),
                "level": activity.get("level", "current_or_next"),
                "goal": activity.get("goal", get_support_tier(state, category_key)),
                "extended_reason": activity.get("extended_reason", ""),
            }

    return None

def ensure_minimum_presence_for_monitor_categories(state: Dict[str, Any], days: Dict[str, Any]) -> Dict[str, Any]:
    """Repair pass so monitor/enrich categories with a target do not disappear."""
    allocation = state.get("weekly_slot_allocation", {})
    target_minutes_by_category = allocation.get("target_minutes_by_category", {})
    child = state["child"]
    daily_time_min = int(child["daily_time_min"])

    assigned_minutes = {k: 0 for k in target_minutes_by_category.keys()}
    for day_name, day_info in days.items():
        for item in day_info.get("items", []):
            k = item.get("category_key")
            if k in assigned_minutes:
                assigned_minutes[k] += int(item.get("duration_min", 0))

    # First try: place one short activity into any day with room.
    for category_key in target_minutes_by_category.keys():
        tier = get_support_tier(state, category_key)
        if tier != "monitor_and_enrich":
            continue
        if assigned_minutes.get(category_key, 0) > 0:
            continue

        bank = state.get("activity_banks", {}).get(category_key, {})
        activities = bank.get("activities", [])
        if not activities:
            continue

        short_candidates = [a for a in activities if int(a.get("duration_min", 5)) <= 5]
        chosen = short_candidates[0] if short_candidates else min(activities, key=lambda a: int(a.get("duration_min", 5)))
        chosen_duration = int(chosen.get("duration_min", 5))

        placed = False
        for day_name, day_info in days.items():
            remaining = daily_time_min - int(day_info.get("total_minutes", 0))
            if remaining >= chosen_duration:
                day_info["items"].append({
                    "category_key": category_key,
                    "category": DOMAIN_CONFIG[category_key]["display"],
                    "title": chosen.get("title"),
                    "instructions": chosen.get("instructions"),
                    "duration_min": chosen_duration,
                    "materials": chosen.get("materials", "common household items"),
                    "level": chosen.get("level", "current_or_next"),
                    "goal": chosen.get("goal", "monitor_and_enrich"),
                })
                day_info["total_minutes"] += chosen_duration
                assigned_minutes[category_key] += chosen_duration
                placed = True
                break

    # Second try: if still missing, swap in one short activity by replacing an activity
    # from a category currently over target.
    for category_key in target_minutes_by_category.keys():
        tier = get_support_tier(state, category_key)
        if tier != "monitor_and_enrich":
            continue
        if assigned_minutes.get(category_key, 0) > 0:
            continue

        bank = state.get("activity_banks", {}).get(category_key, {})
        activities = bank.get("activities", [])
        if not activities:
            continue

        short_candidates = [a for a in activities if int(a.get("duration_min", 5)) <= 5]
        chosen = short_candidates[0] if short_candidates else min(activities, key=lambda a: int(a.get("duration_min", 5)))
        chosen_duration = int(chosen.get("duration_min", 5))

        current_assigned = {}
        for k in target_minutes_by_category.keys():
            current_assigned[k] = 0
        for day_name, day_info in days.items():
            for item in day_info.get("items", []):
                k = item.get("category_key")
                if k in current_assigned:
                    current_assigned[k] += int(item.get("duration_min", 0))

        over_target_categories = [
            k for k in current_assigned.keys()
            if current_assigned[k] > target_minutes_by_category.get(k, 0)
        ]

        swapped = False
        for day_name, day_info in days.items():
            day_items = day_info.get("items", [])
            for idx, item in enumerate(day_items):
                existing_key = item.get("category_key")
                existing_duration = int(item.get("duration_min", 0))
                if existing_key not in over_target_categories:
                    continue
                if existing_duration < chosen_duration:
                    continue

                day_items[idx] = {
                    "category_key": category_key,
                    "category": DOMAIN_CONFIG[category_key]["display"],
                    "title": chosen.get("title"),
                    "instructions": chosen.get("instructions"),
                    "duration_min": chosen_duration,
                    "materials": chosen.get("materials", "common household items"),
                    "level": chosen.get("level", "current_or_next"),
                    "goal": chosen.get("goal", "monitor_and_enrich"),
                }
                day_info["total_minutes"] = sum(int(x.get("duration_min", 0)) for x in day_items)
                swapped = True
                break
            if swapped:
                break

    return days

def build_weekly_schedule(state: Dict[str, Any]) -> Dict[str, Any]:
    if "weekly_slot_allocation" not in state:
        allocate_weekly_slots(state)

    allocation = state["weekly_slot_allocation"]
    daily_time_min = allocation["daily_time_min"]
    target_minutes_by_category = allocation["target_minutes_by_category"]

    days = {
        "day_1": {"items": [], "total_minutes": 0},
        "day_2": {"items": [], "total_minutes": 0},
        "day_3": {"items": [], "total_minutes": 0},
        "day_4": {"items": [], "total_minutes": 0},
        "day_5": {"items": [], "total_minutes": 0},
    }

    soft_floor = state.get("family_guidance_floor", {})

    if not target_minutes_by_category:
        state["weekly_schedule"] = {
            "status": "no_special_support",
            "summary": "No categories need a scheduled weekly activity plan right now.",
            "days": days,
        }
        return state["weekly_schedule"]

    assigned_minutes_by_category = {k: 0 for k in target_minutes_by_category.keys()}
    used_activity_indices = {k: set() for k in target_minutes_by_category.keys()}
    day_names = list(days.keys())

    progress_made = True
    while progress_made:
        progress_made = False

        categories_in_priority_order = sorted(
            target_minutes_by_category.keys(),
            key=lambda k: target_minutes_by_category[k] - assigned_minutes_by_category[k],
            reverse=True,
        )

        for day_name in day_names:
            remaining_day_minutes = daily_time_min - days[day_name]["total_minutes"]
            if remaining_day_minutes <= 0:
                continue

            for category_key in categories_in_priority_order:
                remaining_cat_minutes = (
                    target_minutes_by_category[category_key] - assigned_minutes_by_category[category_key]
                )
                if remaining_cat_minutes <= 0:
                    continue

                bank = state["activity_banks"].get(category_key, {})
                activities = bank.get("activities", [])
                if not activities:
                    continue

                activity = pick_activity_that_fits(
                    activities=activities,
                    used_indices=used_activity_indices[category_key],
                    remaining_minutes=remaining_day_minutes,
                )

                if activity is None:
                    continue

                duration = int(activity.get("duration_min", 5))

                days[day_name]["items"].append({
                    "category_key": category_key,
                    "category": DOMAIN_CONFIG[category_key]["display"],
                    "title": activity.get("title"),
                    "instructions": activity.get("instructions"),
                    "duration_min": duration,
                    "materials": activity.get("materials", "common household items"),
                    "level": activity.get("level", "current_or_next"),
                    "goal": activity.get("goal", get_support_tier(state, category_key)),
                })

                days[day_name]["total_minutes"] += duration
                assigned_minutes_by_category[category_key] += duration
                progress_made = True
                break

    days = ensure_minimum_presence_for_monitor_categories(state, days)

    # recompute assigned minutes after repair pass
    assigned_minutes_by_category = {k: 0 for k in target_minutes_by_category.keys()}
    for day_name, day_info in days.items():
        for item in day_info.get("items", []):
            k = item.get("category_key")
            if k in assigned_minutes_by_category:
                assigned_minutes_by_category[k] += int(item.get("duration_min", 0))

    bonus_activity = pick_weekly_bonus_extended_activity(
        state,
        extended_in_schedule_threshold=15,
        bonus_extended_cap_min=20,
    )

    summary_text = "Weekly schedule built from category activity banks and true daily minute limits."
    status_text = "success"
    if soft_floor.get("enabled"):
        status_text = "family_guidance_floor"
        summary_text = soft_floor.get("summary", "") + " Weekly schedule built as low-intensity enrich-and-observe guidance within the family's time budget."

    schedule = {
        "status": status_text,
        "summary": summary_text,
        "daily_time_min": daily_time_min,
        "target_minutes_by_category": target_minutes_by_category,
        "assigned_minutes_by_category": assigned_minutes_by_category,
        "days": days,
        "weekly_bonus_activity": bonus_activity,
    }

    state["weekly_schedule"] = schedule
    return schedule

def summarizer_agent(state: Dict[str, Any]) -> pd.DataFrame:
    child = state["child"]
    rows = []

    allocation = state.get("weekly_slot_allocation", {}).get("target_minutes_by_category", {})
    concern_profile = ensure_concern_profile(state)
    soft_floor = state.get("family_guidance_floor", {})

    for category_key in DOMAIN_CONFIG:
        delay_info = state["delay_estimates"].get(category_key, {})
        raw_dev_age = state["dev_age"].get(category_key)
        effective_dev_age = get_effective_dev_age(state, category_key)
        metrics = compute_support_metrics(state, category_key)

        chrono_for_gap = min(child.get("chronological_months", 0), 60)
        milestone_gap = None if effective_dev_age is None else max(0, chrono_for_gap - effective_dev_age)

        bank = state.get("activity_banks", {}).get(category_key, {})
        support_tier = metrics["tier"]
        planning_tier = support_tier
        family_guidance_floor_active = False
        family_guidance_summary = ""
        if soft_floor.get("enabled") and soft_floor.get("category_key") == category_key:
            planning_tier = "enrich_and_observe"
            family_guidance_floor_active = True
            family_guidance_summary = soft_floor.get("summary", "")

        rows.append({
            "child_name": child.get("name"),
            "chronological_age_months": child.get("chronological_months"),
            "diagnosis": child.get("diagnosis"),
            "concern": child.get("concern"),
            "daily_time_min": child.get("daily_time_min"),
            "category": DOMAIN_CONFIG[category_key]["display"],
            "estimated_delay_months": delay_info.get("delay_months"),
            "delay_reason": delay_info.get("reason", ""),
            "raw_dev_age_months": raw_dev_age,
            "estimated_dev_age_months": effective_dev_age,
            "effective_dev_age_months": effective_dev_age,
            "milestone_gap_months": milestone_gap,
            "support_tier": support_tier,
            "planning_tier": planning_tier,
            "needs_special_support": support_tier == "needs_special_support",
            "concern_domain_weight": round(concern_profile["domain_weights"].get(category_key, 0.0), 2),
            "concern_subdomain_peak": round(get_category_concern_peak(state, category_key), 2),
            "support_score": metrics["support_score"],
            "weekly_target_minutes_for_category": allocation.get(category_key, 0),
            "activity_bank_status": bank.get("status", "missing"),
            "activity_bank_summary": bank.get("summary", ""),
        })

    return pd.DataFrame(rows)

def display_weekly_schedule(state: Dict[str, Any]) -> None:
    schedule = state.get("weekly_schedule", {})

    print_banner("WEEKLY SCHEDULE")

    if schedule.get("status") == "no_special_support":
        print(schedule.get("summary", "No weekly schedule available."))
        return

    print("Summary:", schedule.get("summary", ""))
    print("Daily time budget:", schedule.get("daily_time_min"))
    print("Target minutes by category:", schedule.get("target_minutes_by_category", {}))
    print("Assigned minutes by category:", schedule.get("assigned_minutes_by_category", {}))

    for day_name, day_info in schedule.get("days", {}).items():
        print("\n" + "-" * 100)
        print(f"{day_name.upper()} — total {day_info.get('total_minutes', 0)} min")
        print("-" * 100)

        items = day_info.get("items", [])
        if not items:
            print("No scheduled activities.")
            continue

        for item in items:
            print(f"- [{item['category']}] {item['title']} ({item['duration_min']} min)")
            print(f"  Goal: {item.get('goal', '')}")
            print(f"  Instructions: {item['instructions']}")
            print(f"  Materials: {item['materials']}")

    bonus = schedule.get("weekly_bonus_activity")
    if bonus:
        print("\n" + "=" * 100)
        print("OPTIONAL WEEKLY BONUS ACTIVITY")
        print("=" * 100)
        print(f"- [{bonus['category']}] {bonus['title']} ({bonus['duration_min']} min)")
        print(f"  Goal: {bonus.get('goal', '')}")
        print(f"  Instructions: {bonus['instructions']}")
        print(f"  Materials: {bonus['materials']}")
        print(f"  Why extended: {bonus.get('extended_reason', '')}")

def run_downstream_planning(state: Dict[str, Any], display_schedule: bool = True) -> pd.DataFrame:
    determine_family_guidance_floor(state)

    for category_key in DOMAIN_CONFIG:
        generate_category_activity_bank(state, category_key)

    allocate_weekly_slots(state)
    build_weekly_schedule(state)

    summary_df = summarizer_agent(state)

    if display_schedule:
        display_weekly_schedule(state)

    return summary_df


In [ ]:

# ------------------------------------------------------------
# Main runners
# ------------------------------------------------------------
def run_all_categories_live():
    """Run one full case in live-answer mode."""
    state = new_state()
    intake_agent_live(state)
    print_child_reference(state)
    print_concern_profile(state)

    delay_agent_all_categories(state)

    for category_key in DOMAIN_CONFIG:
        qna_agent_live(
            state,
            category_key,
            min_yes_confirm=2,
            yes_ratio_confirm=0.60,
        )

    summary_df = run_downstream_planning(state, display_schedule=True)
    return state, summary_df

def run_all_categories_gemini_parent(
    case: Dict[str, Any],
    daily_time_min: int = 10,
    gemini_model: str = "gemini-2.5-flash",
):
    """Run one full case in synthetic-parent mode using Gemini."""
    state = init_state_from_case(case, daily_time_min=daily_time_min)
    print_child_reference(state)
    print_concern_profile(state)

    delay_agent_all_categories(state)

    for category_key in DOMAIN_CONFIG:
        qna_agent_gemini_parent(
            state,
            category_key,
            ask_limit_per_band=3,
            min_yes_confirm=2,
            yes_ratio_confirm=0.60,
            gemini_model=gemini_model,
            delay_between_questions_sec=5,
        )

    summary_df = run_downstream_planning(state, display_schedule=True)
    return state, summary_df

def print_case_reference(case: dict, daily_time_min: int):
    print("\n" + "=" * 110)
    print(f"STARTING {case['case_id']} | {case['child_name']} | {case['age_label']} | {case['diagnosis']}")
    print("=" * 110)
    print(f"Concerns: {case['concerns']}")
    print(f"Daily time: {daily_time_min} min/day")

def run_case_live_from_prefilled_case(
    case: dict,
    default_daily_time_min: int = 10,
    min_yes_confirm: int = 2,
    yes_ratio_confirm: float = 0.60,
):
    """Run a prefilled case live while still asking milestone questions manually."""
    raw_daily = input(
        f"[{case['case_id']}] Daily minutes available? Press Enter for default {default_daily_time_min}: "
    ).strip()
    daily_time_min = int(raw_daily) if raw_daily else int(default_daily_time_min)

    state = init_state_from_case(case, daily_time_min=daily_time_min)
    print_case_reference(case, daily_time_min)
    print_concern_profile(state)

    delay_agent_all_categories(state)

    for category_key in DOMAIN_CONFIG:
        qna_agent_live(
            state,
            category_key,
            min_yes_confirm=min_yes_confirm,
            yes_ratio_confirm=yes_ratio_confirm,
        )

    summary_df = run_downstream_planning(state, display_schedule=True)
    return state, summary_df

def run_case_gemini_from_prefilled_case(
    case: dict,
    daily_time_min: int = 10,
    min_yes_confirm: int = 2,
    yes_ratio_confirm: float = 0.60,
    gemini_model: str = "gemini-2.5-flash",
    delay_between_questions_sec: int = 5,
):
    """Run a prefilled case using Gemini as the parent simulator."""
    state = init_state_from_case(case, daily_time_min=daily_time_min)
    print_case_reference(case, daily_time_min)
    print_concern_profile(state)

    delay_agent_all_categories(state)

    for category_key in DOMAIN_CONFIG:
        qna_agent_gemini_parent(
            state,
            category_key,
            ask_limit_per_band=3,
            min_yes_confirm=min_yes_confirm,
            yes_ratio_confirm=yes_ratio_confirm,
            gemini_model=gemini_model,
            delay_between_questions_sec=delay_between_questions_sec,
        )

    summary_df = run_downstream_planning(state, display_schedule=True)
    return state, summary_df


In [ ]:
# ------------------------------------------------------------
# Test cases
# ------------------------------------------------------------
cases = [
    {"case_id": "C01", "child_name": "Noah", "age_months": 10, "age_label": "10 months",
     "diagnosis": "Down syndrome",
     "concerns": "Not sitting independently yet, Hypotonia, slow feeding, socially engaged"},

    {"case_id": "C02", "child_name": "Maya", "age_months": 18, "age_label": "18 months",
     "diagnosis": "No formal diagnosis",
     "concerns": "No words yet, uses gestures, understands simple commands, good eye contact"},

    {"case_id": "C03", "child_name": "Leo", "age_months": 26, "age_label": "2y 2m",
     "diagnosis": "Autism spectrum disorder",
     "concerns": "Limited eye contact and few words, Repetitive play, transitions hard, picky eating"},

    {"case_id": "C04", "child_name": "Sofia", "age_months": 37, "age_label": "3y 1m",
     "diagnosis": "Prematurity history",
     "concerns": "Speech hard to understand, Fine motor delay, good comprehension"},

    {"case_id": "C05", "child_name": "Ethan", "age_months": 54, "age_label": "4y 6m",
     "diagnosis": "Cerebral palsy",
     "concerns": "Trouble keeping up in preschool, Self-care delays, frequent falls, difficulty with stairs, uses walker sometimes, good cognition"},

    {"case_id": "C06", "child_name": "Amina", "age_months": 60, "age_label": "5 years",
     "diagnosis": "Global developmental delay",
     "concerns": "Learning and routines are hard, Short attention span, follows only simple directions"},

    {"case_id": "C07", "child_name": "Lucas", "age_months": 14, "age_label": "14 months",
     "diagnosis": "Fragile X syndrome",
     "concerns": "Not walking, limited babbling, Sensory sensitivity, shy socially"},

    {"case_id": "C08", "child_name": "Emma", "age_months": 32, "age_label": "2y 8m",
     "diagnosis": "Suspected childhood apraxia of speech",
     "concerns": "Very limited speech output, Uses gestures, gets frustrated, inconsistent sounds"},

    {"case_id": "C09", "child_name": "Zain", "age_months": 48, "age_label": "4 years",
     "diagnosis": "Attention and self-regulation concerns (possible ADHD)",
     "concerns": "Very active, impulsive, Big emotional reactions, interrupts, difficulty following routines, social interest intact"},

    {"case_id": "C10", "child_name": "Isla", "age_months": 44, "age_label": "3y 8m",
     "diagnosis": "SYNGAP1-related disorder",
     "concerns": "Delayed speech and balance, Attention issues, clumsy gait, limited pretend play"},

    {"case_id": "C11", "child_name": "Aria", "age_months": 20, "age_label": "20 months",
     "diagnosis": "No diagnosis",
     "concerns": "Only ~10 words, very active, good eye contact, understands well"},

    {"case_id": "C12", "child_name": "Mateo", "age_months": 30, "age_label": "2y 6m",
     "diagnosis": "No formal diagnosis",
     "concerns": "Good eye contact but no words, lines up toys, responds to name inconsistently"},

    {"case_id": "C13", "child_name": "Luna", "age_months": 42, "age_label": "3y 6m",
     "diagnosis": "Speech delay",
     "concerns": "Limited speech, understands well, parents have very limited time for activities"},

    {"case_id": "C14", "child_name": "Ryan", "age_months": 50, "age_label": "4y 2m",
     "diagnosis": "No diagnosis",
     "concerns": "Tantrums, difficulty with transitions, strong language skills, social but rigid"},
]

df_cases = pd.DataFrame(cases)
display(df_cases)


In [ ]:

# ------------------------------------------------------------
# Advisor-report helpers
# ------------------------------------------------------------
def severity_from_gap(gap_months: float) -> str:
    gap_months = 0 if pd.isna(gap_months) else float(gap_months)
    if gap_months <= 6:
        return "minimal / mild"
    elif gap_months <= 18:
        return "moderate"
    else:
        return "significant"

def split_concerns_to_bullets(concern_text: str, max_items: int = 4):
    items = [x.strip(" -•\n\t") for x in re.split(r"[,\n;]+", str(concern_text)) if x.strip()]
    return items[:max_items]

def _ordered_category_keys_for_pdf(summary_df: pd.DataFrame) -> List[str]:
    if summary_df is None or summary_df.empty:
        return list(DOMAIN_CONFIG.keys())

    work = summary_df.copy()
    work["category_key"] = work["category"].map(
        {v["display"]: k for k, v in DOMAIN_CONFIG.items()}
    )
    tier_col = "planning_tier" if "planning_tier" in work.columns else "support_tier"
    work["tier_rank"] = work[tier_col].map({
        "needs_special_support": 2,
        "monitor_and_enrich": 1,
        "enrich_and_observe": 1,
        "no_special_support": 0,
    }).fillna(0)
    work = work.sort_values(["tier_rank", "milestone_gap_months"], ascending=[False, False])

    ordered = [k for k in work["category_key"].dropna().tolist() if k in DOMAIN_CONFIG]
    seen = set()
    deduped = []
    for k in ordered:
        if k not in seen:
            seen.add(k)
            deduped.append(k)

    for k in DOMAIN_CONFIG.keys():
        if k not in seen:
            deduped.append(k)

    return deduped

def extract_parent_qa_lines(
    state: dict,
    priority_category_keys: Optional[List[str]] = None,
    max_lines_for_pdf: int = 16,
):
    full_lines = []

    ordered_keys = priority_category_keys[:] if priority_category_keys else list(DOMAIN_CONFIG.keys())
    for k in DOMAIN_CONFIG.keys():
        if k not in ordered_keys:
            ordered_keys.append(k)

    for category_key in ordered_keys:
        answers = state.get("qna", {}).get(category_key, [])
        for a in answers:
            milestone = str(a.get("milestone", "")).strip()
            norm_answer = str(a.get("norm_answer", a.get("raw_answer", ""))).strip()
            months = a.get("months", "")
            status = a.get("answer_status", "ok")
            subdomain = a.get("subdomain", "unspecified")
            followup_label = str(a.get("followup_label", "")).strip()
            barrier = str(a.get("performance_barrier", "")).strip()
            extra = ""
            if followup_label:
                extra = f" | followup: {followup_label}"
            if barrier and barrier not in {"", "none"}:
                extra += f" | barrier: {barrier}"
            full_lines.append(f"[{months}m | {subdomain}] {milestone} -> {norm_answer}{extra} ({status})")

    pdf_lines = full_lines[:max_lines_for_pdf]
    return full_lines, pdf_lines

def extract_focus_areas(summary_df: pd.DataFrame, max_items: int = 3):
    if summary_df is None or summary_df.empty:
        return []

    work = summary_df.copy()
    tier_col = "planning_tier" if "planning_tier" in work.columns else "support_tier"
    work = work[work[tier_col].isin(["needs_special_support", "monitor_and_enrich", "enrich_and_observe"])].copy()
    if work.empty:
        return []

    work["tier_rank"] = work[tier_col].map({
        "needs_special_support": 2,
        "monitor_and_enrich": 1,
        "enrich_and_observe": 1,
        "no_special_support": 0,
    })
    sort_col = "milestone_gap_months" if "milestone_gap_months" in work.columns else "estimated_delay_months"
    work = work.sort_values(["tier_rank", sort_col], ascending=[False, False])

    return work["category"].tolist()[:max_items]

def extract_activities_for_case(state: dict, max_activities: int = 5):
    activities = []
    weekly_schedule = state.get("weekly_schedule", {})
    days = weekly_schedule.get("days", {})

    if isinstance(days, dict):
        for day_label, day_info in days.items():
            day_items = day_info.get("items", [])
            for item in day_items:
                activities.append({
                    "day": day_label,
                    "category": item.get("category", item.get("category_display", "")),
                    "title": item.get("title", ""),
                    "description": item.get("instructions", item.get("description", "")),
                    "duration_min": item.get("duration_min", ""),
                    "goal": item.get("goal", item.get("level", item.get("category", ""))),
                })

    if not activities:
        for category_key, bank in state.get("activity_banks", {}).items():
            for item in bank.get("activities", [])[:2]:
                activities.append({
                    "day": "",
                    "category": DOMAIN_CONFIG.get(category_key, {}).get("display", category_key),
                    "title": item.get("title", ""),
                    "description": item.get("instructions", item.get("description", "")),
                    "duration_min": item.get("duration_min", ""),
                    "goal": item.get("goal", item.get("level", "current_or_next")),
                })

    return activities[:max_activities]

def build_case_payload(case: dict, state: dict, summary_df: pd.DataFrame):
    concerns_bullets = split_concerns_to_bullets(case.get("concerns", ""), max_items=4)
    ordered_category_keys = _ordered_category_keys_for_pdf(summary_df)
    parent_qa_full, parent_qa_pdf = extract_parent_qa_lines(
        state,
        priority_category_keys=ordered_category_keys,
        max_lines_for_pdf=16,
    )
    focus_areas = extract_focus_areas(summary_df, max_items=3)
    activities = extract_activities_for_case(state, max_activities=5)

    concern_profile = ensure_concern_profile(state)
    top_concern_subdomains = [x["subdomain"] for x in concern_profile.get("top_subdomains", [])[:4]]

    domain_rows = []
    if summary_df is not None and not summary_df.empty:
        for _, row in summary_df.iterrows():
            gap = row.get("milestone_gap_months", row.get("estimated_delay_months", None))
            domain_rows.append({
                "category": row.get("category", ""),
                "estimated_dev_age_months": row.get("estimated_dev_age_months", ""),
                "gap_months": gap,
                "support_tier": row.get("planning_tier", row.get("support_tier", "")),
                "severity": severity_from_gap(gap if gap is not None else 0),
                "needs_special_support": row.get("needs_special_support", True),
                "summary": row.get("activity_bank_summary", ""),
            })

    support_categories = [r["category"] for r in domain_rows if r.get("support_tier") in {"needs_special_support", "monitor_and_enrich", "enrich_and_observe"}]
    if support_categories:
        summary_text = (
            f"This plan prioritizes {', '.join(support_categories[:3])} for {case['child_name']}. "
            f"The activities are intended to be short, parent-friendly, and aligned with the developmental areas that appeared most delayed or most worth enriching in the interview."
        )
    else:
        family_floor = state.get("family_guidance_floor", {})
        if family_floor.get("enabled"):
            summary_text = family_floor.get("summary", "")
        else:
            summary_text = (
                f"This output did not identify a strong need for special support across the reviewed categories, "
                f"but the family can continue monitoring development and bring remaining questions to the clinician."
            )

    payload = {
        "case_id": case.get("case_id"),
        "child_name": case.get("child_name"),
        "age_label": case.get("age_label"),
        "age_months": case.get("age_months"),
        "diagnosis": case.get("diagnosis"),
        "concerns_bullets": concerns_bullets,
        "daily_time_min": state.get("child", {}).get("daily_time_min", ""),
        "parent_qa_full": parent_qa_full,
        "parent_qa_pdf": parent_qa_pdf,
        "top_concern_subdomains": top_concern_subdomains,
        "domain_rows": domain_rows,
        "focus_areas": focus_areas,
        "activities": activities,
        "summary_text": summary_text,
    }
    return payload

def print_case_screen_summary(payload: dict):
    print("\n" + "=" * 110)
    print(f"CASE {payload['case_id']} | {payload['child_name']} | {payload['age_label']} | {payload['diagnosis']}")
    print("=" * 110)

    print("\n1. CHILD PROFILE")
    print(f"- Name: {payload['child_name']}")
    print(f"- Age: {payload['age_label']} ({payload['age_months']} months)")
    print(f"- Diagnosis: {payload['diagnosis']}")
    print(f"- Daily time available: {payload['daily_time_min']} min/day")
    print("- Key concerns:")
    for c in payload["concerns_bullets"]:
        print(f"  • {c}")
    if payload.get("top_concern_subdomains"):
        print("- Routed concern subdomains:")
        for s in payload["top_concern_subdomains"]:
            print(f"  • {s}")

    print("\n2. PARENT INPUT (full transcript)")
    for line in payload["parent_qa_full"]:
        print(f"- {line}")

    print("\n3. GENEX OUTPUT")
    domain_df = pd.DataFrame(payload["domain_rows"])
    if not domain_df.empty:
        display(domain_df[["category", "estimated_dev_age_months", "gap_months", "support_tier", "severity"]])

    print("\nFocus areas:")
    for x in payload["focus_areas"]:
        print(f"  • {x}")

    print("\nDaily activities:")
    for i, a in enumerate(payload["activities"], start=1):
        print(f"{i}. {a['title']} | {a['category']} | {a['duration_min']} min")
        print(f"   Goal: {a['goal']}")
        print(f"   Description: {a['description']}")

    print("\n4. SUMMARY")
    print(payload["summary_text"])

    print("\n5. ADVISOR REVIEW SECTION")
    print("Rate 1–5:")
    print("- Clinical appropriateness")
    print("- Safety")
    print("- Practicality for parents")
    print("- Clarity")
    print("- Overall usefulness")
    print("Short feedback:")
    print("- What would you change?")
    print("- What is missing?")
    print("- Any concerns?")


In [ ]:
# ------------------------------------------------------------
# PDF rendering + rating sheet
# ------------------------------------------------------------
def wrap_text_to_width(text, font_name, font_size, max_width):
    words = str(text).split()
    if not words:
        return [""]

    lines = []
    current = words[0]
    for word in words[1:]:
        trial = current + " " + word
        if stringWidth(trial, font_name, font_size) <= max_width:
            current = trial
        else:
            lines.append(current)
            current = word
    lines.append(current)
    return lines

def draw_wrapped_text(c, text, x, y, max_width, font_name="Helvetica", font_size=8.5, line_height=10, max_lines=None):
    lines = wrap_text_to_width(text, font_name, font_size, max_width)
    original_lines = lines[:]
    if max_lines is not None:
        lines = lines[:max_lines]
        if len(original_lines) > max_lines and lines:
            lines[-1] = lines[-1][:max(0, len(lines[-1]) - 3)] + "..."
    c.setFont(font_name, font_size)
    for line in lines:
        c.drawString(x, y, line)
        y -= line_height
    return y

def draw_bullets(c, items, x, y, max_width, font_name="Helvetica", font_size=8.5, line_height=10, max_items=None):
    items = list(items)
    if max_items is not None:
        items = items[:max_items]

    for item in items:
        bullet_lines = wrap_text_to_width(str(item), font_name, font_size, max_width - 12)
        if bullet_lines:
            c.drawString(x, y, u"\u2022")
            c.drawString(x + 10, y, bullet_lines[0])
            y -= line_height
            for extra in bullet_lines[1:]:
                c.drawString(x + 10, y, extra)
                y -= line_height
        else:
            y -= line_height
    return y

def draw_box(c, x, y_top, w, h, title):
    c.roundRect(x, y_top - h, w, h, 8, stroke=1, fill=0)
    c.setFont("Helvetica-Bold", 10)
    c.drawString(x + 8, y_top - 14, title)

def render_case_page(c, payload: dict):
    page_w, page_h = landscape(LETTER)
    margin = 0.35 * inch

    left_x = margin
    mid_x = page_w / 2 + 0.10 * inch
    col_w = (page_w / 2) - (1.5 * margin)

    c.setFont("Helvetica-Bold", 15)
    c.drawString(margin, page_h - 0.45 * inch, f"{payload['case_id']} — {payload['child_name']} ({payload['age_label']})")
    c.setFont("Helvetica", 10)
    c.drawString(margin, page_h - 0.68 * inch, f"Diagnosis: {payload['diagnosis']}")

    top_y = page_h - 0.90 * inch
    profile_h = 1.60 * inch
    input_h = 2.55 * inch
    output_h = 2.20 * inch
    activity_h = 2.10 * inch
    summary_h = 0.85 * inch
    rating_h = 1.20 * inch

    draw_box(c, left_x, top_y, col_w, profile_h, "1. Child Profile")
    draw_box(c, left_x, top_y - profile_h - 0.10 * inch, col_w, input_h, "2. Parent Input (questions + answers)")
    draw_box(c, mid_x, top_y, col_w, output_h, "3. Genex Output")
    draw_box(c, mid_x, top_y - output_h - 0.10 * inch, col_w, activity_h, "4. Daily Activities")

    bottom_top = page_h - 6.65 * inch
    full_w = page_w - 2 * margin
    draw_box(c, margin, bottom_top, full_w, summary_h, "5. Summary")
    draw_box(c, margin, bottom_top - summary_h - 0.10 * inch, full_w, rating_h, "6. Advisor Review")

    y = top_y - 28
    c.setFont("Helvetica", 9)
    c.drawString(left_x + 10, y, f"Name: {payload['child_name']}")
    y -= 12
    c.drawString(left_x + 10, y, f"Age: {payload['age_label']} ({payload['age_months']} months)")
    y -= 12
    c.drawString(left_x + 10, y, f"Diagnosis: {payload['diagnosis']}")
    y -= 12
    c.drawString(left_x + 10, y, f"Daily time: {payload['daily_time_min']} min/day")
    y -= 16
    c.setFont("Helvetica-Bold", 9)
    c.drawString(left_x + 10, y, "Key concerns:")
    y -= 11
    y = draw_bullets(c, payload["concerns_bullets"], left_x + 12, y, col_w - 20, max_items=4)

    y = top_y - profile_h - 0.10 * inch - 28
    y = draw_bullets(c, payload["parent_qa_pdf"], left_x + 12, y, col_w - 20, font_size=7.8, line_height=9, max_items=12)
    c.setFont("Helvetica-Oblique", 7.5)
    c.drawString(left_x + 10, top_y - profile_h - input_h - 0.10 * inch + 10, "Full transcript is preserved in JSON output.")

    y = top_y - 28
    domain_rows = payload["domain_rows"][:4]
    c.setFont("Helvetica-Bold", 8.5)
    c.drawString(mid_x + 10, y, "Domain")
    c.drawString(mid_x + 150, y, "Dev age")
    c.drawString(mid_x + 220, y, "Gap")
    c.drawString(mid_x + 270, y, "Tier")
    y -= 11
    c.setFont("Helvetica", 8.2)
    for row in domain_rows:
        c.drawString(mid_x + 10, y, str(row["category"])[:24])
        c.drawString(mid_x + 150, y, str(row["estimated_dev_age_months"]))
        c.drawString(mid_x + 220, y, str(row["gap_months"]))
        c.drawString(mid_x + 270, y, str(row["support_tier"])[:18])
        y -= 10

    y -= 8
    c.setFont("Helvetica-Bold", 9)
    c.drawString(mid_x + 10, y, "Focus areas:")
    y -= 11
    y = draw_bullets(c, payload["focus_areas"], mid_x + 12, y, col_w - 20, max_items=3)

    y = top_y - output_h - 0.10 * inch - 28
    for i, a in enumerate(payload["activities"][:5], start=1):
        line = f"{i}. {a['title']} ({a['duration_min']} min) — {a['goal']}"
        y = draw_wrapped_text(c, line, mid_x + 10, y, col_w - 20, font_name="Helvetica-Bold", font_size=8.2, line_height=9, max_lines=2)
        y = draw_wrapped_text(c, a["description"], mid_x + 16, y, col_w - 26, font_name="Helvetica", font_size=7.8, line_height=8.5, max_lines=2)
        y -= 4

    y = bottom_top - 28
    y = draw_wrapped_text(c, payload["summary_text"], margin + 10, y, full_w - 20, font_name="Helvetica", font_size=8.5, line_height=10, max_lines=4)

    y = bottom_top - summary_h - 0.10 * inch - 28
    rating_lines = [
        "Rate 1–5: Clinical appropriateness | Safety | Practicality for parents | Clarity | Overall usefulness",
        "Short feedback: What would you change? | What is missing? | Any concerns?"
    ]
    for line in rating_lines:
        y = draw_wrapped_text(c, line, margin + 10, y, full_w - 20, font_name="Helvetica", font_size=9, line_height=11, max_lines=2)

    c.showPage()

def build_advisor_rating_sheet(df_cases: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in df_cases.iterrows():
        rows.append({
            "case_id": row["case_id"],
            "child_name": row["child_name"],
            "age_months": row["age_months"],
            "diagnosis": row["diagnosis"],
            "clinical_appropriateness_1to5": "",
            "safety_1to5": "",
            "practicality_for_parents_1to5": "",
            "clarity_1to5": "",
            "overall_usefulness_1to5": "",
            "what_would_you_change": "",
            "what_is_missing": "",
            "any_concerns": "",
        })
    return pd.DataFrame(rows)


In [ ]:

# ------------------------------------------------------------
# Advisor-report export runners
# ------------------------------------------------------------
def run_cases_and_export(
    df_cases: pd.DataFrame,
    mode: str = "live",   # "live" or "gemini"
    output_dir: str = "outputs/advisor_case_pack_v6",
    default_daily_time_min: int = 10,
    pause_between_cases: bool = True,
    gemini_model: str = "gemini-2.5-flash",
):
    """Run all cases and export the advisor packet files."""
    mode = str(mode).strip().lower()
    if mode not in {"live", "gemini"}:
        raise ValueError("mode must be 'live' or 'gemini'")

    out_dir = Path(output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    run_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    combined_pdf_path = out_dir / f"genex_case_report_{mode}_{run_stamp}.pdf"
    rating_csv_path = out_dir / f"advisor_rating_sheet_{mode}_{run_stamp}.csv"
    rating_xlsx_path = out_dir / f"advisor_rating_sheet_{mode}_{run_stamp}.xlsx"
    json_path = out_dir / f"case_payloads_{mode}_{run_stamp}.json"

    c = canvas.Canvas(str(combined_pdf_path), pagesize=landscape(LETTER))
    all_payloads = []
    rating_df = build_advisor_rating_sheet(df_cases)

    for i, (_, row) in enumerate(df_cases.iterrows(), start=1):
        case = row.to_dict()

        if mode == "live":
            state, summary_df = run_case_live_from_prefilled_case(
                case=case,
                default_daily_time_min=default_daily_time_min,
                min_yes_confirm=2,
                yes_ratio_confirm=0.60,
            )
        else:
            state, summary_df = run_case_gemini_from_prefilled_case(
                case=case,
                daily_time_min=default_daily_time_min,
                min_yes_confirm=2,
                yes_ratio_confirm=0.60,
                gemini_model=gemini_model,
                delay_between_questions_sec=5,
            )

        payload = build_case_payload(case, state, summary_df)
        all_payloads.append(payload)

        print_case_screen_summary(payload)
        render_case_page(c, payload)

        per_case_summary_path = out_dir / f"{case['case_id']}_{case['child_name']}_summary.csv"
        summary_df.to_csv(per_case_summary_path, index=False)

        if pause_between_cases and i < len(df_cases):
            input("\nPress Enter to continue to the next case...")

    c.save()

    rating_df.to_csv(rating_csv_path, index=False)
    rating_df.to_excel(rating_xlsx_path, index=False)

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(all_payloads, f, indent=2)

    print("\n" + "=" * 110)
    print("DONE")
    print("=" * 110)
    print("Mode:", mode)
    print("Combined PDF:", combined_pdf_path)
    print("Rating CSV :", rating_csv_path)
    print("Rating XLSX:", rating_xlsx_path)
    print("Payload JSON:", json_path)

    return all_payloads, rating_df, combined_pdf_path

def run_cases_and_export_in_batches(
    df_cases: pd.DataFrame,
    batch_size: int = 5,
    gap_between_batches_sec: int = 120,
    mode: str = "gemini",
    output_dir: str = "outputs/advisor_case_pack_v6",
    default_daily_time_min: int = 10,
    pause_between_cases: bool = False,
    gemini_model: str = "gemini-2.5-flash",
):
    """Run advisor export in batches, useful for Gemini quota stability."""
    mode = str(mode).strip().lower()
    if mode not in {"live", "gemini"}:
        raise ValueError("mode must be 'live' or 'gemini'")

    out_dir = Path(output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    run_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    combined_pdf_path = out_dir / f"genex_case_report_{mode}_{run_stamp}.pdf"
    rating_csv_path = out_dir / f"advisor_rating_sheet_{mode}_{run_stamp}.csv"
    rating_xlsx_path = out_dir / f"advisor_rating_sheet_{mode}_{run_stamp}.xlsx"
    json_path = out_dir / f"case_payloads_{mode}_{run_stamp}.json"

    c = canvas.Canvas(str(combined_pdf_path), pagesize=landscape(LETTER))
    all_payloads = []
    rating_df = build_advisor_rating_sheet(df_cases)
    total_cases = len(df_cases)

    for start_idx in range(0, total_cases, batch_size):
        end_idx = min(start_idx + batch_size, total_cases)
        batch_df = df_cases.iloc[start_idx:end_idx]

        print("\n" + "=" * 110)
        print(f"RUNNING BATCH {start_idx // batch_size + 1} | cases {start_idx + 1} to {end_idx} of {total_cases}")
        print("=" * 110)

        for _, row in batch_df.iterrows():
            case = row.to_dict()

            if mode == "live":
                state, summary_df = run_case_live_from_prefilled_case(
                    case=case,
                    default_daily_time_min=default_daily_time_min,
                    min_yes_confirm=2,
                    yes_ratio_confirm=0.60,
                )
            else:
                state, summary_df = run_case_gemini_from_prefilled_case(
                    case=case,
                    daily_time_min=default_daily_time_min,
                    min_yes_confirm=2,
                    yes_ratio_confirm=0.60,
                    gemini_model=gemini_model,
                    delay_between_questions_sec=5,
                )

            payload = build_case_payload(case, state, summary_df)
            all_payloads.append(payload)

            print_case_screen_summary(payload)
            render_case_page(c, payload)

            per_case_summary_path = out_dir / f"{case['case_id']}_{case['child_name']}_summary.csv"
            summary_df.to_csv(per_case_summary_path, index=False)

            if pause_between_cases:
                input("\nPress Enter to continue to the next case...")

        if end_idx < total_cases:
            print("\n" + "-" * 110)
            print(f"Batch complete. Waiting {gap_between_batches_sec} seconds before next batch...")
            print("-" * 110)
            time.sleep(gap_between_batches_sec)

    c.save()
    rating_df.to_csv(rating_csv_path, index=False)
    rating_df.to_excel(rating_xlsx_path, index=False)

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(all_payloads, f, indent=2)

    print("\n" + "=" * 110)
    print("DONE")
    print("=" * 110)
    print("Mode:", mode)
    print("Combined PDF:", combined_pdf_path)
    print("Rating CSV :", rating_csv_path)
    print("Rating XLSX:", rating_xlsx_path)
    print("Payload JSON:", json_path)

    return all_payloads, rating_df, combined_pdf_path


## Sanity check (optional)

Run this before the example cells if you want a quick check that the core functions are loaded, including the new concern-router and v6 support-tier logic.


In [ ]:
print("run_all_categories_live" in globals())
print("run_all_categories_gemini_parent" in globals())
print("run_downstream_planning" in globals())
print("generate_category_activity_bank" in globals())
print("allocate_weekly_slots" in globals())
print("build_weekly_schedule" in globals())
print("summarizer_agent" in globals())
print("concern_router" in globals())
print("get_support_tier" in globals())

## Example run cells

Use **only one** of the cells below at a time.


In [ ]:
# Example 1 — one fully live run
# state, summary_df = run_all_categories_live()
# display(summary_df)


In [ ]:
# Example 2 — one synthetic case using Gemini as the parent
case = df_cases.iloc[0].to_dict()
state, summary_df = run_all_categories_gemini_parent(
    case,
    daily_time_min=10,
    gemini_model="gemini-2.5-flash"
)
display(summary_df)


In [ ]:
# V9 test case 1 — Emma (speech/apraxia stress test)
case = next(c for c in cases if c["case_id"] == "C08")

state, summary_df = run_all_categories_gemini_parent(
    case,
    daily_time_min=10,
    gemini_model="gemini-2.5-flash",
)

display(summary_df)
display_weekly_schedule(state)


In [ ]:
case_ids = ["C05", "C08", "C11", "C14"]

results = {}

for case_id in case_ids:
    print(f"\n### Running case {case_id}")

    case = next(c for c in cases if c["case_id"] == case_id)

    state, summary_df = run_all_categories_gemini_parent(
        case,
        daily_time_min=10,
        gemini_model="gemini-2.5-flash",
    )

    results[case_id] = {
        "state": state,
        "summary_df": summary_df
    }

    display(summary_df)
    display_weekly_schedule(state)


In [ ]:
# Example 3A — export advisor packet in live mode
# all_payloads, advisor_rating_df, pdf_path = run_cases_and_export(
#     df_cases=df_cases,
#     mode="live",
#     output_dir="outputs/advisor_case_pack_v7",
#     default_daily_time_min=10,
#     pause_between_cases=True,
# )
# display(advisor_rating_df.head())
# pdf_path


In [ ]:
# Example 3B — export advisor packet in Gemini synthetic-parent mode, in batches of 5 with 2 min gap
# all_payloads, advisor_rating_df, pdf_path = run_cases_and_export_in_batches(
#     df_cases=df_cases,
#     batch_size=5,
#     gap_between_batches_sec=120,
#     mode="gemini",
#     output_dir="outputs/advisor_case_pack_v13",
#     default_daily_time_min=10,
#     pause_between_cases=False,
#     gemini_model="gemini-2.5-flash",
# )
# display(advisor_rating_df.head())
# pdf_path


# How We Designed This Notebook and How the Current Algorithm Works (v13)

We built this notebook as a structured prototype of the Genex developmental interview and home-support engine for children ages 0-5. Our goal is to keep the logic transparent enough for expert review while still making the system practical enough to become a real product workflow.

## What changed in v13

Version 13 starts from the more stable **v11** notebook rather than from v12.

We kept the core v11 logic:
- subdomain-aware concern routing
- split internal language scoring
- continuous support score
- broad safety profile
- confirmed / emerging / not_demonstrated band framework
- family-guidance floor
- playdate / group activities moved out of the core weekly schedule and into Optional Weekly Bonus Activity

The main new addition in v13 is a **lightweight parent follow-up layer** for ambiguous milestone answers:

- if a parent answers **sometimes**, we ask **why only sometimes**
- if a parent answers **with_help**, we ask **what kind of help is needed**
- if a parent answers **no**, we ask **what kind of “no” this is**

These follow-up answers are stored in the transcript as review information and barrier context, but in v13 they do **not automatically override milestone scoring**. Developmental age is still driven by the normalized milestone answer itself.

## Why we made this change

The main app will be answered by **parents**, not by Gemini. In live parent use, follow-up answers can provide clinically useful context without forcing the system to rely on more brittle heuristic suppression rules.

Version 13 intentionally avoids the heavier spillover guardrails from v12. The design principle here is:

> milestone evidence still drives developmental age, while follow-up answers help explain *why* a skill is inconsistent, assisted, or absent.

This makes the system easier to defend and less dependent on hardcoded behavioral keyword rules.

## Core pipeline

The shared pipeline is:

1. collect or load the child profile  
2. route diagnosis/concern text into weighted **subdomain** signals  
3. build a broad **safety/practical constraint profile** from the same intake text  
4. estimate a starting delay by developmental domain  
5. select milestone questions by combining developmental-age proximity with concern/subdomain relevance  
6. ask milestone questions and, only when needed, ask a short follow-up for **sometimes / with_help / no**  
7. convert the answers into month-band evidence summaries  
8. classify each month band as **confirmed**, **emerging**, or **not_demonstrated**  
9. derive a developmental age per domain from the highest sufficiently supported band  
10. compute a continuous support score and assign a support tier  
11. if all tiers are **no_special_support**, activate the **family guidance floor** in the strongest concern-relevant category  
12. generate category activity banks using diagnosis-aware and safety-aware guardrails  
13. build a realistic weekly schedule from the parent's available time  
14. optionally export a one-page case summary and advisor rating sheet

## Why the v13 change is useful

In a real parent-facing product, the answer labels **sometimes**, **with_help**, and **no** often hide important context. Version 13 captures some of that context in a very lightweight way, but stops short of using aggressive hardcoded suppression rules. That lets us compare v11 and v13 more fairly while keeping the system generalizable and easier to explain to advisors.


# Genex Advisor Review Handout  

## Developmental Interview + Home Support Prototype (Ages 0-5, v13)

Thank you for reviewing this Genex prototype.

This packet contains sample child cases produced by our current developmental interview and home-support planning system for children ages 0-5. At this stage, our goal is to evaluate whether the logic, developmental estimates, concern-routing behavior, support scoring, and suggested home activities feel clinically reasonable, practical, and safe.

### What is new in v13
Version 13 starts from the more stable v11 notebook and adds one lightweight product feature:
- if a parent answers **sometimes**, **with_help**, or **no**, the system asks one short follow-up
- that follow-up is stored in the transcript as barrier/context information
- the follow-up helps distinguish emerging skill from distraction, prompting, refusal, emotional support, or physical help
- however, in v13 these follow-up answers do **not automatically overwrite milestone scoring**

### What this prototype is trying to do
This prototype is designed to:
- take a child profile and parent concern
- ask structured milestone questions
- estimate developmental level across key domains
- assign one of three core support tiers
- still give families a practical home plan even when no category meets the threshold for scheduled support
- record a small amount of extra context when milestone answers are ambiguous

It is **not diagnostic** and does **not replace clinical judgment**.

Version 13 also keeps the v11 product rule that the core weekly schedule should stay focused on short, repeatable, at-home activities, while playdates, park meetups, playground peer practice, and other context-dependent activities stay in **Optional Weekly Bonus Activity** only.


# Short Cover Note

**Dear Advisor,**

Thank you for reviewing this early Genex prototype.

This packet contains sample child cases generated through our current developmental interview and home-support planning workflow for ages 0-5. Version 13 starts from the more stable v11 system and adds one lightweight parent-facing improvement: when a milestone answer is **sometimes**, **with_help**, or **no**, the system asks one short follow-up so the transcript can capture barrier/context information such as inconsistency, prompting, physical help, refusal, or distraction.

In version 13, those follow-up answers are stored for review, but they do **not automatically override milestone scoring**. We made that choice to keep the system more generalizable and easier to defend while still learning from richer parent input.

This is still a prototype and is not intended to diagnose or replace clinical judgment. We are using this review process to understand where the system is working well, where it is too aggressive or too conservative, and what should be revised before further development.

Your feedback on appropriateness, clarity, practicality, safety, the subdomain structure, the language scoring split, the emerging-band logic, the family-guidance floor, and the new follow-up question layer would be incredibly valuable.

**Best,**  
**Sara / Genex**
